In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as scipy_norm, laplace as scipy_laplace, t as scipy_t
from scipy.special import digamma
from scipy.optimize import brentq
from mpl_toolkits.mplot3d import Axes3D

import sys
sys.path.extend(["../../"])

from utils import heteroscedastic_loss_function, computation_graph_heteroscedastic_gaussian, computation_graph_heteroscedastic_variance, grad_heteroscedastic_loss_wrt_linear_model
from utils import fit_ols_heteroscedastic, computation_graph_linear, create_computation_graph_linear, create_computation_graph_linear_from_ols
from utils import squared_loss_function, absolute_loss_function, student_t_loss_function, student_t_full_loss_function
from utils import grad_absolute_loss_wrt_linear_model, grad_student_t_loss_wrt_linear_model
from utils import fit_norm2_least_square, student_t_e_step, student_t_m_step_w, student_t_m_step_sigma2, student_t_m_step_nu
from utils_data import generate_poly_features

$$
\newcommand{\lvec}{\mathbf{l}}
\newcommand{\xvec}{\mathbf{x}}
\newcommand{\xvect}{\mathbf{x}^T}
\newcommand{\Xmat}{\mathbf{X}}
\newcommand{\Xmatt}{\mathbf{X}^T}
\newcommand{\Amat}{\mathbf{A}}
\newcommand{\Amatt}{\mathbf{A}^T}
\newcommand{\avec}{\mathbf{a}}
\newcommand{\avect}{\mathbf{a}^T}
\newcommand{\Bmat}{\mathbf{B}}
\newcommand{\Bmatt}{\mathbf{B}^T}
\newcommand{\Cmat}{\mathbf{C}}
\newcommand{\cvec}{\mathbf{c}}
\newcommand{\Cmatt}{\mathbf{C}^T}
\newcommand{\tvec}{\mathbf{t}}
\newcommand{\tvect}{\mathbf{t}^T}
\newcommand{\Tmat}{\mathbf{T}}
\newcommand{\Tmatt}{\mathbf{T}^T}
\newcommand{\yvec}{\mathbf{y}}
\newcommand{\Ymat}{\mathbf{Y}}
\newcommand{\Ymatt}{\mathbf{Y}^T}
\newcommand{\Zmat}{\mathbf{Z}}
\newcommand{\zvec}{\mathbf{z}}
\newcommand{\wvec}{\mathbf{w}}
\newcommand{\wvect}{\mathbf{w}^T}
\newcommand{\wvecsigma}{\mathbf{w}_\sigma}
\newcommand{\wvecsigmat}{\mathbf{w}_{\sigma}^T}
\newcommand{\sigmatwovec}{\boldsymbol{\sigma^2}}
\newcommand{\Wmat}{\mathbf{W}}
\newcommand{\Wmatt}{\mathbf{W}^T}
\newcommand{\Wmatsigma}{\mathbf{W}_\sigma}
\newcommand{\Wmatsigmat}{\mathbf{W}_{\sigma}^T}
\newcommand{\Vmat}{\mathbf{V}}
\newcommand{\Vmatt}{\mathbf{V}^T}
\newcommand{\Vvec}{\mathbf{v}}
\newcommand{\Vvect}{\mathbf{v}^T}
\newcommand{\Imat}{\mathbf{I}}
\newcommand{\Sigmainv}{\Sigma^{-1}}
\newcommand{\onevec}{\mathbf{1}}
\newcommand{\onevect}{\mathbf{1}^T}
\newcommand{\dd}{\mathrm{d}}
\newcommand{\diag}{\text{diag}}
\newcommand{\pareinv}[1]{\left(#1\right)^{-1}}
\newcommand{\pare}[1]{\left(#1\right)}
\newcommand{\pareT}[1]{\left(#1\right)^{T}}
\renewcommand{\bra}[1]{\left[#1\right]}
\newcommand{\braT}[1]{\left[#1\right]^{T}}
\newcommand{\tr}[1]{\text{tr}\left(#1\right)}
\newcommand{\vvec}{\text{vec} }
\newcommand{\sgn}{\operatorname{sgn}}
$$

# Probabilistic Perspective

Linear models, as the ones we have studied so far, can be framed through many different perspectives. In my opinion, one of the most elegant ways to frame these models are through the lens of probability theory. Why? Well in essence when we model data, we are trying to model the scenario that has generate this data. This scenario is uncertain, ie it has some uncertainty associated. Probability theory is the mathematical tool we use to model this uncertainty.

As we will see now, many of the loss functions we use are implicitely associated with some probability distribution we are assuming about how our data has been generated. This has some advantages, since knowing the assumed probability distribution help us in making reliable decisions when something does not work as we want, or when we know some specific properties of our data.

## Maximum Likelihood estimation.

Maximum likelihood is a method for estimating the parameters of a probability distribution, given some observed data. The idea relies on assuming that our data follows some distribution, and we want to find the parameters of this distribution that are more likely to have generated this data.

For instance, consider some data $x\in\mathbb{R}$. We might have $N$ points $x^n$. Maximum likelihood solves the following optimization problem:

$$
\begin{split}
\argmax_\theta p( \bra{x^1,x^2,\dots,x^N} \mid \theta) = \argmax_\theta \prod^N_{n=1} p(x^n\mid\theta)
\end{split}
$$


Importantly, we usually take the $\log$ likelihood function. Why? Because the log is a monotonic transformation that does not change the solution to the optimization problem but has the following advantages:

* It results in more stable numerical computations, since the product os small terms quickly go to zero.
* The product becomes a sum, allowing for stochastic optimization methods without bias.


So in practice we optimize:

$$
\begin{split}
 \argmax_\theta \log \prod^N_{n=1} p(x^n\mid\theta) = \argmax_\theta  \sum^N_{n=1} \log p(x^n\mid\theta)
\end{split}
$$

### Example: Maximum  Likelihood Estimation for a Gaussian distribution

Maximum Likelihood Estimation for a Gaussian distribution is obtained by a coordinate ascent procedure: we optimize the log likelihood with respect to one parameter while keeping the rest fixed, and we repeat this for every parameter, iterating until convergence.

Consider $N$ i.i.d. observations $x^1,\dots,x^N \in \mathbb{R}$, each one assumed to be generated from a Gaussian distribution with unknown mean $\mu$ and variance $\sigma^2$:

$$
\begin{split}
p(x^n\mid \mu,\sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\pare{-\frac{(x^n-\mu)^2}{2\sigma^2}}
\end{split}
$$

Following what we have just seen, the log likelihood of the full dataset $\theta = \bra{\mu,\sigma^2}$ is:

$$
\begin{split}
\log p(\bra{x^1,\dots,x^N}\mid \mu,\sigma^2) &= \sum^N_{n=1} \log p(x^n\mid \mu,\sigma^2)\\
&= \sum^N_{n=1} \pare{ -\frac{1}{2}\log(2\pi) - \frac{1}{2}\log\sigma^2 - \frac{(x^n-\mu)^2}{2\sigma^2} }\\
&= -\frac{N}{2}\log(2\pi) - \frac{N}{2}\log\sigma^2 - \frac{1}{2\sigma^2}\sum^N_{n=1}(x^n-\mu)^2
\end{split}
$$

**Step 1: optimize with respect to $\mu$, keeping $\sigma^2$ fixed.**

The first two terms do not depend on $\mu$, so this is equivalent to minimizing $\sum_n (x^n-\mu)^2$, ie the squared loss we already used to fit our linear models. Taking the derivative and setting it to zero:

$$
\begin{split}
\frac{\partial}{\partial \mu} \log p(\bra{x^1,\dots,x^N}\mid \mu,\sigma^2) &= \frac{1}{\sigma^2}\sum^N_{n=1}(x^n-\mu) = 0\\
\sum^N_{n=1} x^n &= N\mu\\
\mu_{\text{MLE}} &= \frac{1}{N}\sum^N_{n=1} x^n
\end{split}
$$

The maximum likelihood estimate of the mean is, unsurprisingly, the sample mean.

**Step 2: optimize with respect to $\sigma^2$, keeping $\mu$ fixed.**


$$
\begin{split}
\frac{\partial}{\partial \sigma^2} \log p(\bra{x^1,\dots,x^N}\mid \mu,\sigma^2) &= -\frac{N}{2\sigma^2} + \frac{1}{2\pare{\sigma^2}^2}\sum^N_{n=1}(x^n-\mu)^2 = 0\\
\frac{N}{2\sigma^2} &= \frac{1}{2\pare{\sigma^2}^2}\sum^N_{n=1}(x^n-\mu)^2\\
 \sigma^2 &= \frac{1}{N}\sum^N_{n=1}(x^n-\mu)^2
\end{split}
$$

Plugging back $\mu_{\text{MLE}}$ from Step 1, the maximum likelihood estimate of the variance is:

$$
\begin{split}
\sigma^2_{\text{MLE}} = \frac{1}{N}\sum^N_{n=1}(x^n-\mu_{\text{MLE}})^2
\end{split}
$$

**Note on coordinate ascent.** In general, coordinate ascent needs several iterations across parameters until convergence. In this particular case, however, $\mu_{\text{MLE}}$ (Step 1) does not depend on $\sigma^2$ at all, so plugging it directly into Step 2 already gives the joint optimum: coordinate ascent converges in a single pass over $\mu$ and $\sigma^2$. This is a special property of the Gaussian likelihood, and it is exactly the same reason why, later on, we will be able to solve for $\wvec_{\text{MLE}}$ and $\sigma^2_{\text{MLE}}$ separately once we introduce the design matrix $\Xmat$.


In [ ]:
%matplotlib inline
plt.close("all")

# true parameters used to generate synthetic data
mu_true = 2.0
sigma_true = 1.5
N = 50

np.random.seed(0)
x = np.random.normal(mu_true, sigma_true, size=N)

# MLE estimates, following the closed form we derived above
mu_mle = np.mean(x)
sigma2_mle = np.mean((x - mu_mle)**2)

print(f"mu_MLE = {mu_mle:.3f} (true mu = {mu_true})")
print(f"sigma^2_MLE = {sigma2_mle:.3f} (true sigma^2 = {sigma_true**2:.3f})")

# log likelihood as a function of mu, with sigma^2 fixed at its MLE value
def log_likelihood(mu, sigma2, x):
    N = x.shape[0]
    return -N/2*np.log(2*np.pi) - N/2*np.log(sigma2) - (1/(2*sigma2)) * np.sum((x - mu)**2)

mu_grid = np.linspace(mu_mle - 3, mu_mle + 3, 200)
ll_grid = np.array([log_likelihood(mu, sigma2_mle, x) for mu in mu_grid])

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(mu_grid, ll_grid, color='k', label=r'$\log p(\{x^n\} \mid \mu, \sigma^2_{MLE})$')
ax.axvline(mu_mle, color='C1', linestyle='--', label=fr'$\mu_{{MLE}} = {mu_mle:.3f}$')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel('log likelihood')
ax.set_title(r'Log likelihood as a function of $\mu$, with $\sigma^2=\sigma^2_{MLE}$ fixed')
ax.legend()

In [ ]:
# now let both mu and sigma vary, and look at the log likelihood surface jointly
mu_range = np.linspace(mu_mle - 1, mu_mle + 1, 100)
sigma_range = np.linspace(0.5, 2.5, 100)
Mu, Sigma = np.meshgrid(mu_range, sigma_range)

LL = np.zeros_like(Mu)
for i in range(Mu.shape[0]):
    for j in range(Mu.shape[1]):
        LL[i, j] = log_likelihood(Mu[i, j], Sigma[i, j]**2, x)

# the raw log likelihood is dominated by how bad things get for very small sigma
# (dividing by a tiny sigma^2 makes it blow up), which drowns out the curvature
# near the optimum. To see the shape near the maximum, we look at the *relative*
# log likelihood (0 at the optimum) and clip everything below vmin: those regions
# are astronomically less likely anyway, so flattening them does not hide anything
# interesting.
LL_rel = LL - LL.max()
vmin = -30
levels = np.linspace(vmin, 0, 40)
LL_clipped = np.clip(LL_rel, vmin, 0)

# 2D view: contourf
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
cf = ax.contourf(Mu, Sigma, LL_clipped, levels=levels, cmap='viridis', extend='min')
ax.plot(mu_mle, np.sqrt(sigma2_mle), 'x', color='red', markersize=12, markeredgewidth=3,
        label=r'$(\mu_{MLE},\sigma_{MLE})$')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\sigma$')
ax.set_title(r'Log likelihood $\log p(\{x^n\}\mid\mu,\sigma^2)$, contour (relative to the max)')
fig.colorbar(cf, ax=ax)
ax.legend()

# 3D view: surface
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(Mu, Sigma, LL_clipped, cmap='viridis', vmin=vmin, vmax=0, linewidth=0, antialiased=True)
ax.set_zlim(vmin, 0)
ax.scatter([mu_mle], [np.sqrt(sigma2_mle)], [0], color='red', s=60, label=r'$(\mu_{MLE},\sigma_{MLE})$')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\sigma$')
ax.set_zlabel('log likelihood (relative to the max)')
ax.set_title(r'Log likelihood $\log p(\{x^n\}\mid\mu,\sigma^2)$, surface (relative to the max)')

## Maximum A Posterior Estimation.

Another way to fit parameters of a models is through the maximum a posterior estimation. From Bayes theorem we now:

$$
\begin{split}
p(\theta \mid \bra{x^1,x^2,\dots,x^N} ) = \frac{ p( \bra{x^1,x^2,\dots,x^N} \mid \theta) p(\theta)}{ p( \bra{x^1,x^2,\dots,x^N})}
\end{split}
$$

Now if we find the parameter that maximizes the log posterior distribution, we have:


$$
\begin{split}
& \argmax_\theta \log p(\theta \mid \bra{x^1,x^2,\dots,x^N} ) \\
&= \argmax_\theta \log  p( \bra{x^1,x^2,\dots,x^N} \mid \theta) + \log p(\theta) -\log p( \bra{x^1,x^2,\dots,x^N})\\
&= \argmax_\theta \log  p( \bra{x^1,x^2,\dots,x^N} \mid \theta) + \log p(\theta) 
\end{split}
$$

because $p(\bra{x^1,x^2,\dots,x^N})$ does not depend on $\theta$, hence the derivative when optimizing is zero.


### Example: Maximum A Posteriori Estimation for a Gaussian distribution

Consider again $N$ i.i.d. observations $x^1,\dots,x^N$ generated from a Gaussian distribution with known variance $\sigma^2$ and unknown mean $\mu$. Unlike in the MLE case, we now place a prior distribution over $\mu$, encoding our belief about it before seeing any data. Let's use a Gaussian prior:

$$
\begin{split}
p(\mu) = \frac{1}{\sqrt{2\pi\tau^2}}\exp\pare{-\frac{(\mu-\mu_0)^2}{2\tau^2}}
\end{split}
$$

where $\mu_0$ and $\tau^2$ are hyperparameters that we fix beforehand, and that encode how much, and around which value, we believe $\mu$ should concentrate.

From what we just showed, the MAP estimate solves $\argmax_\mu \log p(\bra{x^1,\dots,x^N}\mid \mu,\sigma^2) + \log p(\mu)$. Plugging in both Gaussian densities and using the log likelihood we already derived for the MLE example:

$$
\begin{split}
&\log p(\bra{x^1,\dots,x^N}\mid\mu,\sigma^2) + \log p(\mu) = \\
&-\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum^N_{n=1}(x^n-\mu)^2 -\frac{1}{2}\log(2\pi\tau^2) - \frac{1}{2\tau^2}(\mu-\mu_0)^2
\end{split}
$$

Dropping the terms that do not depend on $\mu$, the MAP problem becomes:

$$
\begin{split}
\argmax_\mu \log p(\bra{x^1,\dots,x^N}\mid \mu,\sigma^2) + \log p(\mu) = \argmin_\mu \sum^N_{n=1}(x^n-\mu)^2 + \frac{\sigma^2}{\tau^2}(\mu-\mu_0)^2
\end{split}
$$

Taking the derivative and setting it to zero:

$$
\begin{split}
\frac{\partial}{\partial \mu}\bra{ \sum^N_{n=1}(x^n-\mu)^2 + \frac{\sigma^2}{\tau^2}(\mu-\mu_0)^2 } &= -2\sum^N_{n=1}(x^n-\mu) + \frac{2\sigma^2}{\tau^2}(\mu-\mu_0) = 0\\
N\mu + \frac{\sigma^2}{\tau^2}\mu &= \sum^N_{n=1}x^n + \frac{\sigma^2}{\tau^2}\mu_0\\
\mu\pare{N + \frac{\sigma^2}{\tau^2}} &= \sum^N_{n=1}x^n + \frac{\sigma^2}{\tau^2}\mu_0\\
\mu_{\text{MAP}} &= \frac{\sum^N_{n=1}x^n + \frac{\sigma^2}{\tau^2}\mu_0}{N+\frac{\sigma^2}{\tau^2}}
\end{split}
$$

Recalling $\mu_{\text{MLE}} = \frac{1}{N}\sum^N_{n=1} x^n$, we can rewrite $\mu_{\text{MAP}}$ as a weighted average between the sample mean and the prior mean:

$$
\begin{split}
\mu_{\text{MAP}} = \frac{N}{N+\sigma^2/\tau^2}\,\mu_{\text{MLE}} + \frac{\sigma^2/\tau^2}{N+\sigma^2/\tau^2}\,\mu_0
\end{split}
$$

Two limits are worth noting:

* As $\tau^2 \to \infty$ (an uninformative, flat prior), the second term vanishes and $\mu_{\text{MAP}} \to \mu_{\text{MLE}}$: with no real prior information, MAP reduces to MLE.
* As $N \to \infty$ the first term dominates, and again $\mu_{\text{MAP}}\to\mu_{\text{MLE}}$: with enough data, the likelihood overwhelms whatever prior we placed.

**Connection with $L_2$ regularization.** Set $\mu_0=0$, ie a prior centered at zero. Going back to the log-posterior *before* we cleared any denominators:

$$
\begin{split}
\log p(\bra{x^1,\dots,x^N}\mid\mu,\sigma^2) + \log p(\mu) = -\frac{1}{2\sigma^2}\sum^N_{n=1}(x^n-\mu)^2 - \frac{1}{2\tau^2}\mu^2 + \text{const}
\end{split}
$$

so the MAP problem, without rescaling anything, is:

$$
\begin{split}
\argmin_\mu \underbrace{\frac{1}{2\sigma^2}\sum^N_{n=1}(x^n-\mu)^2}_{C(\mu)} + \underbrace{\frac{1}{2\tau^2}\mu^2}_{\lambda\mu^2}
\end{split}
$$

This has exactly the general form $C(\mu)+\lambda\mu^2$ we derive in full generality in the "Probabilistic Interpretation of Regularizers" section later in this book, with $C(\mu)$ the genuine negative log-likelihood (Gaussian noise, hence the $\frac{1}{2\sigma^2}$) and regularization coefficient $\lambda=\frac{1}{2\tau^2}$, depending only on the prior's own variance $\tau^2$. Crucially, $\sigma^2$ does **not** enter $\lambda$: it is a property of the likelihood (how noisy the data is), not of the prior, and it correctly stays inside $C(\mu)$ rather than leaking into the regularization strength. A stronger regularization ($\lambda$ large) corresponds to a more confident, narrower prior (small $\tau^2$); a weaker regularization corresponds to a wider, less informative prior.

If we instead want to solve this MAP problem with the already-implemented `fit_norm2_least_square(X,T,lam)`, which minimizes the *raw*, unscaled sum of squares $\sum_n(x^n-\mu)^2 + \text{lam}\cdot\mu^2$ rather than the noise-scaled NLL above, we must multiply our objective by $2\sigma^2$ to strip the $\frac{1}{2\sigma^2}$ off the data term:

$$
\argmin_\mu \sum^N_{n=1}(x^n-\mu)^2 + \frac{\sigma^2}{\tau^2}\mu^2
$$

This is still the same minimizer (multiplying by a positive constant never moves the argmin), so the value to pass as `lam` is $\sigma^2/\tau^2$ — not because the regularization coefficient itself depends on $\sigma^2$, but purely because `fit_norm2_least_square`'s convention operates on the raw SSE rather than the precision-weighted NLL. The function itself needs no change: it is a general-purpose ridge solver that deliberately makes no probabilistic assumptions, and translating a probabilistic $\lambda$ into its `lam` argument is exactly this kind of bookkeeping, standard whenever one connects a Bayesian derivation back to a plain SSE-plus-penalty solver.

## Homoscedastic Gaussian Observation Model


### Single Output Model

Once we have introduced the concepts of maximum likelihood and maximum a posteriori distribution, we can establish the connection between what we have seen so far and the assumed probability distributions.


If we remember, in our regression model, we had:

$$
\begin{split}
y = \xvect\wvec
\end{split}
$$


When this model is optimized through the square loss:

$$
\begin{split}
\sum^N_{n=1} (t^n-y^n)^2
\end{split}
$$

We are implicitly assuming that our target $t$ follows a Gaussian distribution with mean given by $\xvect\wvec$, a constant variance across points (homoscedastic model), and we find $\wvec$ that maximizes the log-likelihood. To see so, let's reuse the above formulation. We now have:


$$
\begin{split}
p(t^n\mid {\xvect}^{(n)}\wvec,\sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\pare{-\frac{(t^n-{\xvect}^{(n)}\wvec)^2}{2\sigma^2}}
\end{split}
$$

So the log-likelihood function can be written down as:

$$
\begin{split}
&\sum^N_{n=1}\log \frac{1}{\sqrt{2\pi\sigma^2}}\exp\pare{-\frac{(t^n-{\xvect}^{(n)}\wvec)^2}{2\sigma^2}}\\
&= \sum^N_{n=1} \pare{ -\frac{1}{2}\log(2\pi) - \frac{1}{2}\log\sigma^2 - \frac{(t^n-{\xvect}^{(n)}\wvec)^2}{2\sigma^2} }\\
&= -\frac{N}{2}\log(2\pi) - \frac{N}{2}\log\sigma^2 - \frac{1}{2\sigma^2}\sum^N_{n=1}(t^n-{\xvect}^{(n)}\wvec)^2
\end{split}
$$

Maximizing the log-likelihood wrt $\wvec$  requires taking the gradient of the log-likelihood function wrt $\wvec$. So all the terms that do not depend on $\wvec$ can be dropped, since their derivative is zero. Thus:

$$
\begin{split}
&\argmax_\wvec \sum^N_{n=1}\log \frac{1}{\sqrt{2\pi\sigma^2}}\exp\pare{-\frac{(t^n-{\xvect}^{(n)}\wvec)^2}{2\sigma^2}}\\
&= \argmax_\wvec - \frac{1}{2\sigma^2}\sum^N_{n=1}(t^n-{\xvect}^{(n)}\wvec)^2\\
&= \argmin_\wvec \frac{1}{2\sigma^2}\sum^N_{n=1}(t^n-{\xvect}^{(n)}\wvec)^2
\end{split}
$$


Since $\frac{1}{2\sigma^2}$ is just a scaling constant that does not depend on $\wvec$ it does not change where the optimum occurs. This means that this is equivalent to minimizing:

$$
\argmin_\wvec \sum^N_{n=1}(t^n-{\xvect}^{(n)}\wvec)^2
$$

which is the sum of squared errors. The optimal parameter is given by the OLS solution, as we know:

$$
\wvec_\text{opt} = \pareinv{\Xmatt\Xmat}\Xmatt\tvec
$$


Now, given the optimal mean, we know the optimal variance is given by:

$$
\begin{split}
\sigma^2_{\text{opt}} = \frac{1}{N}\sum^N_{n=1}(t^n-{\xvect}^{(n)}\wvec_\text{opt})^2
\end{split}
$$

#### The role of the variance

$\sigma_2$ represents the aleatoric uncertainty, because it is the noise intrinsic to the data. The one we cannot reduce by collecting more data. The epistemic uncertainty is encoded in the posterior distribution of the weight vector given the data and some prior.

This noise is also referred to as the variance of the distribution of the residual. 
 
In econometrics, one of the assumptions we need to make for the ordinary least squares error to be valid is that the noise is zero mean, common to any point, and its expected value does not depend on the data. 

The probabilistic perspective starts by assuming some likelihood of the outputs given the inputs, and works out a loss function from maximizing the log likelihood or log posterior. Modelling data through this approach allows us to create new losses that might have some interesting properties.

From this perspective, we can easily understand why in the previous chapter we generated data through:

$$
\begin{split}
t = \xvect\wvec + \epsilon; \quad \epsilon  \sim \mathcal{N}(\epsilon|0,\sigma^2_\epsilon)
\end{split}
$$

This also justifies why, if the model is well specified (ie our data is actually drawn from some line with Gaussian noise), the distribution of the residuals is zero-mean Gaussian with common variance $\mathcal{N}\pare{t-y\mid 0,\sigma^2_\epsilon}$. In other words, when you fit a linear model, you can plot the difference between your optimal predictions and the target outcomes. If that distribution is Gaussian, then you are probably in the right direction.



#### Visualization

Let's visualize the probabilistic model we have been describing. At each $\xvec$ we observe a Gaussian distribution with mean given by $\xvect\wvec$ and constant variance $\sigma^2$

In [ ]:
%matplotlib inline
plt.close("all")

# Illustrate the Gaussian observation model: for a given x, the target t is
# assumed to be Gaussian distributed around the regression line y = x^T w,
# with a constant variance sigma^2 across all points (homoscedastic).

w_true = np.array([1.0, 0.8])  # [w_0, w_1]
sigma_true = 0.6

x_line = np.linspace(0, 5, 200)
y_line = w_true[0] + w_true[1] * x_line

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
ax.plot(x_line, y_line, color='k', linewidth=2, label=r'$y = \mathbf{x}^T\mathbf{w}$')

# at a few x locations, draw the Gaussian density sideways, centered on the line
x_points = [1.0, 2.5, 4.0]
bump_scale = 1.0  # purely visual, controls how wide the bumps are drawn

for x0 in x_points:
    mean = w_true[0] + w_true[1] * x0
    t_range = np.linspace(mean - 3 * sigma_true, mean + 3 * sigma_true, 200)
    density = np.exp(-0.5 * (t_range - mean)**2 / sigma_true**2) / np.sqrt(2 * np.pi * sigma_true**2)
    ax.plot(x0 + bump_scale * density, t_range, color='C1')
    ax.fill_betweenx(t_range, x0, x0 + bump_scale * density, color='C1', alpha=0.3)
    ax.axvline(x0, color='gray', linestyle=':', linewidth=1)

ax.set_xlabel('$x$')
ax.set_ylabel('$t$')
ax.set_title(r'Gaussian observation model: $p(t\mid x,\mathbf{w},\sigma^2) = \mathcal{N}(t \mid \mathbf{x}^T\mathbf{w}, \sigma^2)$')
ax.legend()

### Multioutput model

Let's now extend the observation model to $C$ outputs. Each data point $n$ has a target vector $\tvec^{(n)}\in\mathbb{R}^C$ (row $n$ of $\Tmat$), and predictions are given by $\Ymat=\Xmat\Wmat$, exactly as we saw in the multioutut model. Let's connect the loss function we used there to the likelihood being assumed. If we remember, the loss function being used was the squared distance between all datapoints and all the outputs:

$$
\begin{split}
L(\Wmat,\Xmat,\Tmat) &= \sum^N_{n=1}\sum^C_{c=1}(t_c^n - y_c^n)^2
\end{split}
$$

#### Common variance per output

We assume the $C$-dimensional target vector $\tvec^{(n)}$ follows a multivariate Gaussian with mean given by the model's prediction and covariance matrix $\Sigma=\sigma^2\Imat$, ie an independent Gaussian with the same variance $\sigma^2$ on every output:

$$
\begin{split}
p(\tvec^{(n)}\mid \xvec^{(n)},\Wmat,\sigma^2) = \mathcal{N}\pare{\tvec^{(n)}\mid \Wmatt\xvec^{(n)}, \sigma^2\Imat}
\end{split}
$$

Note that $\pareT{{\xvect}^{(n)}\Wmat}=\Wmatt\xvec^{(n)}$: transposing a product reverses the order of the factors and transposes each one. We need to write the mean this way, as $\Wmatt\xvec^{(n)}$, because the mean of a multivariate Gaussian must be a column vector matching the shape of $\tvec^{(n)}$, and ${\xvect}^{(n)}\Wmat$ (the row-$n$ form we use for $\Ymat=\Xmat\Wmat$) is a row vector, not a column vector.

Since independent random variables have a joint density that factorizes as the product of their marginals, this is the same as writing:

$$
\begin{split}
\mathcal{N}\pare{\tvec^{(n)}\mid \Wmatt\xvec^{(n)}, \sigma^2\Imat} = \prod^C_{c=1}\mathcal{N}\pare{t_c^n\mid {\xvect}^{(n)}\wvec_c,\sigma^2}
\end{split}
$$

where $\wvec_c$ denotes column $c$ of $\Wmat$. Since the $N$ data points are also i.i.d, the full likelihood is:

$$
\begin{split}
p(\Tmat\mid\Xmat,\Wmat,\sigma^2) = \prod^N_{n=1}\prod^C_{c=1}\mathcal{N}\pare{t_c^n\mid {\xvect}^{(n)}\wvec_c,\sigma^2}
\end{split}
$$

Taking the log turns the double product into a double sum:

$$
\begin{split}
\log p(\Tmat\mid\Xmat,\Wmat,\sigma^2) &= \sum^N_{n=1}\sum^C_{c=1} \log \mathcal{N}\pare{t_c^n\mid {\xvect}^{(n)}\wvec_c,\sigma^2}\\
&= \sum^N_{n=1}\sum^C_{c=1}\pare{-\frac12\log(2\pi)-\frac12\log\sigma^2-\frac{(t_c^n-{\xvect}^{(n)}\wvec_c)^2}{2\sigma^2}}\\
&= -\frac{NC}{2}\log(2\pi) - \frac{NC}{2}\log\sigma^2 - \frac{1}{2\sigma^2}\sum^N_{n=1}\sum^C_{c=1}(t_c^n-{\xvect}^{(n)}\wvec_c)^2
\end{split}
$$

Dropping the terms that do not depend on $\Wmat$, exactly as we did in the single output case:

$$
\begin{split}
&\argmax_\Wmat \log p(\Tmat\mid\Xmat,\Wmat,\sigma^2) = \\
&\argmin_\Wmat \sum^N_{n=1}\sum^C_{c=1}(t_c^n-{\xvect}^{(n)}\wvec_c)^2 
\end{split}
$$

recovering exactly the loss $L(\Wmat,\Xmat,\Tmat)$ we already know.

##### Coordinate update for the variance: common variance per output

Here, the covariance matrix of the joint Gaussian is given by the scalar matrix product $\sigma^2\Imat$. While I am quite sure we could compactly express everything using multivariate calculus through the lens of the matrix-normal distribution, I will leave that as an exercise for the reader to get extra points in the subject.

Here, however, we just follow the simpler approach that starts from the log-likelihood above but expresses the joint distribution over $C$ outputs compactly. Note:

$$
\begin{split}
&\log p(\Tmat\mid\Xmat,\Wmat,\sigma^2) = \\ 
& \log \prod^N_{n=1}\prod^C_{c=1} \frac{1}{\sqrt{2\pi\sigma^2}}\exp\pare{-\frac{(t_c^n-{\xvect}^{(n)}\wvec_c)^2}{2\sigma^2}}\\
&= \sum^N_{n=1} \log \mathcal{N}\pare{\tvec^{(n)}\mid \Wmatt\xvec^{(n)},\sigma^2\Imat}\\
&= -\frac{NC}{2}\log(2\pi) -\frac12\bra{NC\log\sigma^2 + \frac{1}{\sigma^2}\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}
\end{split}
$$

Starting with the differential over $\sigma^2$ (the first term is constant wrt $\sigma^2$ and drops out):

$$
\begin{split}
&\dd\bra{ -\frac12\bra{NC\log\sigma^2 + \frac{1}{\sigma^2}\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}}\\
&= -\frac12\bra{NC\frac{\dd\sigma^2}{\sigma^2} - \frac{1}{\sigma^4}\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\dd\sigma^2}\\
&= \bra{-\frac{NC}{2\sigma^2} + \frac{1}{2\sigma^4}\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}} \dd\sigma^2
\end{split}
$$

Here the gradient coincides with the Jacobian, and so we can directly equate to $0$:

$$
\begin{split}
&-\frac{NC}{2\sigma^2} + \frac{1}{2\sigma^4}\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}} = 0 \\
&\sigma^2_\text{opt} = \frac{1}{NC}\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\\
&= \frac{1}{NC}\sum^N_{n=1}\sum^C_{c=1}(t_c^n-{\xvect}^{(n)}\wvec_c)^2
\end{split}
$$

As expected, this is not more than the sample variance obtained through all the outputs and datapoints.


#### Independent per output variance

So far we have assumed a shared variance per output. However, if we still assume independent outputs, each one with its own variance, the result is the same.

Now each output has its own noise variance $\sigma_c^2$ (still homoscedastic across data points within an output, but heteroscedastic across outputs). The log-likelihood becomes:

$$
\begin{split}
\log p(\Tmat\mid\Xmat,\Wmat,\bra{\sigma_c^2}) = -\frac{NC}{2}\log(2\pi) - \frac{N}{2}\sum^C_{c=1}\log\sigma_c^2 - \sum^C_{c=1}\frac{1}{2\sigma_c^2}\sum^N_{n=1}(t_c^n-{\xvect}^{(n)}\wvec_c)^2
\end{split}
$$

Dropping the terms that do not depend on $\Wmat$ (including $\sigma_c^2$, since it only rescales each column's term by a positive constant that does not move where it is minimized):

$$
\begin{split}
&\argmax_\Wmat \log p(\Tmat\mid\Xmat,\Wmat,\bra{\sigma_c^2}) = \\
&\argmin_\Wmat \sum^N_{n=1}\sum^C_{c=1}(t_c^n-{\xvect}^{(n)}\wvec_c)^2
\end{split}
$$

recovering exactly the same loss $L(\Wmat,\Xmat,\Tmat)$ as with a common variance.

The natural question is: how is it possible that a different statistical model yields exactly the same solution? Well, that's not true at all. If we consider the optimal variance parameter per model, we do observe differences. To see this, we need to compute the derivative wrt the scalar $\sigma^2$ and the derivative wrt the vector $\sigmatwovec$. It turns out that we can use differentials to obtain the Jacobians easily, and from these obtain the derivative and the resulting coordinate update.


##### Coordinate update for the variance

Here, the covariance matrix of the joint Gaussian is given by the vector matrix product $\sigmatwovec\Imat$. The canonical form of the first-order differential of a scalar-value vector argument function is:

$$
\begin{split}
\dd y = J \dd\xvec; 
\end{split}
$$

with J being a row vector. Here, the likelihood still factorizes across $C$. However, we will represent everything in vector form, rather than deriving a particular expression per $\sigma^2_c$. Here we can probably use the Matrix Normal distribution. However, the derivations not using it are a bit notation intense so here I might not recommend studying that direction before getting a deep inside into this one. We start from the log likelihood:

$$
\begin{split}
&\log p(\Tmat\mid\Xmat,\Wmat,\sigmatwovec) = \\ 
& \log \prod^N_{n=1}\prod^C_{c=1} \frac{1}{\sqrt{2\pi\sigma_c^2}}\exp\pare{-\frac{(t_c^n-{\xvect}^{(n)}\wvec_c)^2}{2\sigma_c^2}}\\
&= \sum^N_{n=1} \log \mathcal{N}\pare{\tvec^{(n)}\mid \Wmatt\xvec^{(n)},\sigmatwovec\Imat}\\
&= -\frac{NC}{2}\log(2\pi) -\frac{N}{2}\log\left|\sigmatwovec\Imat\right| - \frac12\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\sigmatwovec\Imat}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
\end{split}
$$

where the last expression comes from applying the log to the expression of the multivariate Gaussian distribution. Working out the differential requires applying some rules that relate the diagonal operator with the trace. 

$$
\begin{split}
&\dd \log p(\Tmat\mid\Xmat,\Wmat,\sigmatwovec) =\\
&\dd\bra{-\frac{NC}{2}\log(2\pi) -\frac{N}{2}\log\left|\sigmatwovec\Imat\right| - \frac12\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\sigmatwovec\Imat}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}\\
&= \dd\bra{-\frac{N}{2}\log\left|\diag\pare{\sigmatwovec}\right| - \frac12\sum^N_{n=1}\tr{\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag\pare{\sigmatwovec}}}}
\end{split}
$$

For ease of derivation, we derive the differentials of the sum part by parts. Using the R' matrix from section 5 in Minka (https://tminka.github.io/papers/matrix/minka-matrix.pdf), we write:

$$
\begin{split}
\dd\log\left|\diag(\sigmatwovec)\right| &= \tr{\pareinv{\diag(\sigmatwovec)}\dd\bra{\diag(\sigmatwovec)}}\\
&= \vvec\braT{\pareinv{\diag\pare{\sigmatwovec}}}\dd\vvec{\diag\pare{\sigmatwovec}}\\
&= \vvec\braT{\pareinv{\diag\pare{\sigmatwovec}}}R'\dd\sigmatwovec
\end{split}
$$

Now the second part of the differential:

$$
\begin{split}
&\dd\bra{-\frac12\sum^N_{n=1}\tr{\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag(\sigmatwovec)}}}\\
&= -\frac12\sum^N_{n=1}\tr{\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\dd\pareinv{\diag(\sigmatwovec)}}\\
&= \frac12\sum^N_{n=1}\tr{\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag(\sigmatwovec)}\dd\bra{\diag(\sigmatwovec)}\pareinv{\diag(\sigmatwovec)}}\\
&= \frac12\sum^N_{n=1}\tr{\pareinv{\diag(\sigmatwovec)}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag(\sigmatwovec)}\dd\bra{\diag(\sigmatwovec)}}\\
&= \frac12\sum^N_{n=1}\vvec\braT{\pareinv{\diag(\sigmatwovec)}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag(\sigmatwovec)}}R'\dd\sigmatwovec\\
\end{split}
$$

So the total differential (applying the transposed version of identity 67 from Minka):

$$
\begin{split}
\dd \log p(\Tmat\mid\Xmat,\Wmat,\sigmatwovec) &= -\frac{N}{2}\vvec\braT{\pareinv{\diag(\sigmatwovec)}}R'\dd\sigmatwovec + \frac12\sum^N_{n=1}\vvec\braT{\pareinv{\diag(\sigmatwovec)}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag(\sigmatwovec)}}R'\dd\sigmatwovec\\
&= \bra{-\frac{N}{2}\diag^{-1}\bra{\pareinv{\diag(\sigmatwovec)}}^T + \frac12\sum^N_{n=1}\diag^{-1}\bra{\pareinv{\diag(\sigmatwovec)}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag(\sigmatwovec)}}^T}\dd\sigmatwovec\\
\end{split}
$$

Now, we can identify the Jacobian as the term multiplying $\dd\sigmatwovec$. The gradient is its transpose, which is given by:

$$
\nabla_{\sigmatwovec}\log p(\Tmat\mid\Xmat,\Wmat,\sigmatwovec) = -\frac{N}{2}\diag^{-1}\bra{\pareinv{\diag(\sigmatwovec)}} + \frac12\sum^N_{n=1}\diag^{-1}\bra{\pareinv{\diag(\sigmatwovec)}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareinv{\diag(\sigmatwovec)}}
$$

Now, note that the operator $\diag^{-1}$ extracts the diagonal of the matrix (equation 57 in Minka). Also, the inverse of a diagonal matrix is just the inverse of the elements one by one. This can be applied to the first term in the sum, giving $\sigmatwovec^{-1}$, which is defined as the elementwise inverse of the vector $\sigmatwovec$. On the second term, something similar can be applied. First note that two diagonal matrices, $\pareinv{\diag(\sigmatwovec)}$, are multiplying the dense matrix $\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}$, one on each side. The $\diag^{-1}$ of this product is the Hadamard product between $\sigmatwovec^{-1}\circ\sigmatwovec^{-1}$ and the diagonal of the dense matrix, $\diag^{-1}\bra{\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}$. This gives the vector equation representing the gradient:

$$
\nabla_{\sigmatwovec}\log p(\Tmat\mid\Xmat,\Wmat,\sigmatwovec) = -\frac{N}{2}\sigmatwovec^{-1} + \frac12\sum^N_{n=1}\sigmatwovec^{-1}\circ\sigmatwovec^{-1}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
$$

Optimal $\sigmatwovec$ is obtained by equating this gradient to zero.

$$
\begin{split}
0 &= -\frac{N}{2}\sigmatwovec^{-1} + \frac12\sum^N_{n=1}\sigmatwovec^{-1}\circ\sigmatwovec^{-1}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
\end{split}
$$

Multiplying both sides (Hadamard) by $\sigmatwovec\circ\sigmatwovec$, and using $\sigmatwovec^{-1}\circ\sigmatwovec=\onevec$:

$$
\begin{split}
0 &= \sigmatwovec\circ\sigmatwovec \circ\bra{-\frac{N}{2}\sigmatwovec^{-1} + \frac12\sum^N_{n=1}\sigmatwovec^{-1}\circ\sigmatwovec^{-1}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}\\
0 &= -\frac{N}{2}\sigmatwovec + \frac12\sum^N_{n=1}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\\
\sigmatwovec_\text{opt} &= \frac1N\sum^N_{n=1}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\circ\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
\end{split}
$$

As expected, this is the per-output sample variance: the $c$-th component of $\sigmatwovec_\text{opt}$ is exactly $\frac1N\sum^N_{n=1}(t_c^n-{\xvect}^{(n)}\wvec_c)^2$.



#### Full dependence

So far, we have established the connection between the sum of squared errors and the loss function being minimized. We have seen that our loss function implies that the data being modelled follows a Gaussian or Multivariate Gaussian with the mean given by $\xvect\Wmat$. In the [multioutput regression section](1_Regression_Shallow_theory.ipynb), we mentioned that the loss function associated with multioutput regression considers independence between outputs. We have, in fact, shown that we are considering a joint independent Gaussian distribution.

Thus, the final step is deriving the loss function when we want to consider dependent outputs. Here, we will see that we cannot obtain the solution in two coordinate steps, one for the mean and one for the covariance, since the equations get coupled. This is something similar to what happens with K-means. If we denote with $\Sigma$ the covariance matrix between the outputs, then the log-likelihood is given by:

$$
\begin{split}
&\log p(\Tmat\mid\Xmat,\Wmat,\sigmatwovec) = \\ 
&= \sum^N_{n=1} \log \mathcal{N}\pare{\tvec^{(n)}\mid \Wmatt\xvec^{(n)},\Sigma}\\
&= -\frac{NC}{2}\log(2\pi) -\frac{N}{2}\log\left|\Sigma\right| - \frac12\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
\end{split}
$$


Dropping the terms that do not depend on $\Wmat$, and multiplicative constants, exactly as we did before:

$$
\begin{split}
&\argmax_\Wmat \log p(\Tmat\mid\Xmat,\Wmat,\Sigma) = \\
&\argmin_\Wmat \sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
\end{split}
$$

Unlike the previous cases, this is not the loss $L(\Wmat,\Xmat,\Tmat)$ anymore. When $\Sigma$ is not diagonal, $\Sigmainv$ mixes the $C$ outputs inside each term of the sum, so the loss can no longer be written as a sum of independent per-output squared errors: it is a genuine Mahalanobis-distance loss, weighting the residual $\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}$ according to the full output covariance structure. This could let us think that optimal $\Wmat$ now depends on the value of $\Sigma$. This would imply that, as in the K-means algorithm, several iterations of a coordinate descent routine are needed.

We can show, however, that $\Sigma$ does not play any role in the optimal value for $\Wmat$.


This is a scalar-valued, matrix argument function. We know the first order differential is given by $\dd y = J\dd \vvec {\Wmat}$. Using the differential of the quadratic form, and the chain rule (see example 1 in https://arxiv.org/pdf/2506.23996 section 2.6, or section 4.1), the differential is given by:

$$
\begin{split}
\dd\bra{\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}} &= \sum^N_{n=1}2\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\,\dd\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\\
&= \sum^N_{n=1}2\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\,\bra{-\bra{\Imat\otimes{\xvect}^{(n)}}\dd\vvec\Wmat}\\
&= -2\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv \bra{\Imat \otimes {\xvect}^{(n)}}\dd\vvec \Wmat
\end{split}
$$


The Jacobian is the row vector multiplying $\dd\vvec\Wmat$:

$$
J = -2\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\bra{\Imat\otimes{\xvect}^{(n)}}
$$

The gradient is its transpose, using $\bra{\Imat\otimes{\xvect}^{(n)}}^T=\Imat\otimes\xvec^{(n)}$:

$$
\nabla_{\vvec\Wmat} = -2\sum^N_{n=1}\bra{\Imat\otimes\xvec^{(n)}}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
$$

Equating to zero:

$$
0 = \sum^N_{n=1}\bra{\Imat\otimes\xvec^{(n)}}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
$$

Let's try and solve for $\Wmat$. First note that $\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}$ is a column vector, so we can write $\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}} =\vvec\bra{\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}$. Let's call this vector $\avec$. Also $\vvec\bra{\avec}=\vvec\bra{\avect}$. Then we have:

$$
\begin{split}
\bra{\Imat\otimes\xvec^{(n)}}\vvec\bra{\avect}
\end{split}
$$

Applying $\vvec\bra{\Amat\Xmat\Bmat}=\bra{\Bmatt\otimes\Amat}\vvec\bra{\Xmat}$ to kronecker products we can write:

$$
\begin{split}
\bra{\Imat\otimes\xvec^{(n)}}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}} &= \vvec\bra{\xvec^{(n)}\pareT{\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}}\\
 &= \vvec\bra{\xvec^{(n)}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv}
\end{split}
$$

Now we can write our equation to be solved, which, right-multiplying by $\Sigma$ removes it from the equation:

$$
\begin{split}
0 &= \sum^N_{n=1}\vvec\bra{\xvec^{(n)}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv}\\
0 &= \sum^N_{n=1}\vvec\bra{\xvec^{(n)}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}
\end{split}
$$

which again shows that ordinary least squares is the solution to the problem.

##### Coordinate update for the variance.

The coordinate update for the covariance is pretty simple, at least simpler than in the previous case. Picking the differential of the log likelihood wrt $\Sigma$, we have:

$$
\begin{split}
\dd\log p(\Tmat\mid\Xmat,\Wmat,\Sigma) &= \dd\bra{-\frac{NC}{2}\log(2\pi) -\frac{N}{2}\log\left|\Sigma\right| - \frac12\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}}\\
&= -\frac{N}{2}\dd\log\left|\Sigma\right| - \frac12\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\dd\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
\end{split}
$$

Using $\dd\log\left|\Sigma\right|=\tr{\Sigmainv\dd\Sigma}$ and $\dd\Sigmainv=-\Sigmainv\dd\Sigma\,\Sigmainv$:

$$
\dd\log p(\Tmat\mid\Xmat,\Wmat,\Sigma) = -\frac{N}{2}\tr{\Sigmainv\dd\Sigma} + \frac12\sum^N_{n=1}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\dd\Sigma\,\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
$$

The second term is already a scalar, so we can wrap it in a trace for free, and use the cyclic property to bring $\dd\Sigma$ to the right:

$$
\begin{split}
\dd\log p(\Tmat\mid\Xmat,\Wmat,\Sigma) &= -\frac{N}{2}\tr{\Sigmainv\dd\Sigma} + \frac12\sum^N_{n=1}\tr{\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\dd\Sigma}\\
&= \tr{\bra{-\frac{N}{2}\Sigmainv + \frac12\sum^N_{n=1}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv}\dd\Sigma}
\end{split}
$$

The bracketed term is symmetric (a sum of $\Sigmainv$ and $\Sigmainv u u^T\Sigmainv$ terms, both symmetric since $\Sigma$ is symmetric), so converting the trace to canonical vec form:

$$
\dd\log p(\Tmat\mid\Xmat,\Wmat,\Sigma) = \vvec\braT{-\frac{N}{2}\Sigmainv + \frac12\sum^N_{n=1}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv}\dd\vvec\Sigma
$$

The Jacobian is the row vector multiplying $\dd\vvec\Sigma$, and since the bracketed term is symmetric, the gradient (its transpose) is the same expression without the outer transpose:

$$
\nabla_{\vvec\Sigma} = -\frac{N}{2}\Sigmainv + \frac12\sum^N_{n=1}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv
$$

Equating to zero, and left- and right-multiplying by $\Sigma$ to remove $\Sigmainv$ from both sides:

$$
\begin{split}
0 &= -\frac{N}{2}\Sigmainv + \frac12\sum^N_{n=1}\Sigmainv\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\Sigmainv\\
0 &= -\frac{N}{2}\Sigma + \frac12\sum^N_{n=1}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\\
\Sigma_\text{opt} &= \frac1N\sum^N_{n=1}\pare{\tvec^{(n)}-\Wmatt\xvec^{(n)}}\pareT{\tvec^{(n)}-\Wmatt\xvec^{(n)}}
\end{split}
$$

As expected, this is exactly the sample covariance matrix of the residuals evaluated at $\Wmat_\text{opt}$.


## Heteroscedastic Gaussian Observation Model

So far, we have seen observation models where the variance $\sigma^2$ is constant across points $\xvec^{n}$. We have only considered dependencies in multioutput models. Heteroscedastic regression represents models in which the variance varies across locations. Here is an example. We are simulating sales in a shop vs the temperature in the street. Sales here can be negative when we do not sell enough to compensate for the investment of that day (salaries and so on)

In [ ]:
rng = np.random.default_rng(7)
n = 300
temperature_hetero = rng.uniform(-20, 30, size=n)

w_true, b_true = 0.15, 0.5
mean_sales_hetero = w_true * temperature_hetero + b_true  # linear mean, instead of softplus

sigma0, sigma1 = 0.1, 1.0
sigma_sales = sigma0 + sigma1 * np.abs(mean_sales_hetero)  # heteroscedastic: noise scale grows with |mean|

sales_hetero = mean_sales_hetero + rng.normal(0, 1, size=n) * sigma_sales

plt.figure(figsize=(7, 5))
plt.scatter(temperature_hetero, sales_hetero, marker="x")
plt.xlabel("temperature")
plt.ylabel("sales")
plt.title("Heteroscedastic data to be modeled: sales vs. temperature (linear mean)")
plt.grid(True)
plt.show()

How do we model this data?. Well, again, through an observation model which we consider to be Gaussian. Before, we were considering a Gaussian observation model with common variance per point, which gives the likelihood:

$$
\begin{split}
\prod^N_{n=1} p(t^n\mid {\xvect}^{(n)}\wvec,\sigma^2) = \prod^N_{n=1}\frac{1}{\sqrt{2\pi\sigma^2}}\exp\pare{-\frac{(t^n-{\xvect}^{(n)}\wvec)^2}{2\sigma^2}}
\end{split} 
$$

The idea here is to turn this model into one in which the variance depends on the point location $n$. We would start by setting the likelihood:

$$
\begin{split}
\prod^N_{n=1} p(t^n\mid {\xvect}^{(n)}\wvec,\sigma^2) = \prod^N_{n=1}\frac{1}{\sqrt{2\pi\sigma_n^2}}\exp\pare{-\frac{(t^n-{\xvect}^{(n)}\wvec)^2}{2\sigma_n^2}}
\end{split} 
$$

Since we know independent Gaussians come from a joint Gaussian, we can compactly express this likelihood as:


$$
\begin{split}
\prod^N_{n=1} p(t^n\mid {\xvect}^{(n)}\wvec,\sigma_n^2) = \mathcal{N}\pare{\tvec\mid \Xmat\wvec,\Vmat}, \quad \Vmat=\diag\pare{\sigma_1^2,\dots,\sigma_N^2}
\end{split} 
$$

The log-likelihood is given by:

$$
\log p(\tvec\mid\Xmat,\wvec,\Vmat) = -\frac{N}{2}\log(2\pi) - \frac12\log\left|\Vmat\right| - \frac12\pareT{\tvec-\Xmat\wvec}\Vmat^{-1}\pare{\tvec-\Xmat\wvec}
$$

As always, maximizing the log likelihood wrt $\wvec$ results in:

$$
\begin{split}
&\argmax_\wvec \log p(\tvec\mid\Xmat,\wvec,\Vmat) = \\
&\argmax_\wvec -\frac{N}{2}\log(2\pi) - \frac12\log\left|\Vmat\right| - \frac12\pareT{\tvec-\Xmat\wvec}\Vmat^{-1}\pare{\tvec-\Xmat\wvec}=\\
&\argmin_\wvec \pareT{\tvec-\Xmat\wvec}\Vmat^{-1}\pare{\tvec-\Xmat\wvec}\\
&\argmin_\wvec \sum^N_{n=1}\frac{1}{\sigma^2_n}\pare{t^{(n)}-\xvec^{(n)}\wvec}^2\\
\end{split}
$$

While we could expand this quadratic form into a sum of terms, since $\Vmat$ is diagonal we will workout explicitely the minimum of this. To do so, we, again, work out the differential. Applying some standard rules from the differential, we have:

$$
\begin{split}
\dd\bra{\pareT{\tvec-\Xmat\wvec}\Vmat^{-1}\pare{\tvec-\Xmat\wvec}}=-2\pareT{\tvec-\Xmat\wvec}\Vmat^{-1}\Xmat\dd\wvec
\end{split}
$$

This identifies the Jacobian, from which the gradient can be obtained by transposition; setting it to zero yields optimal $\wvec$:

$$
\begin{split}
&\nabla_\wvec = 0\\
&-2\Xmatt\Vmat^{-1}\pare{\tvec-\Xmat\wvec} = 0\\
&\Xmatt\Vmat^{-1}\tvec=\Xmatt\Vmat^{-1}\Xmat\wvec\\
&\wvec_\text{opt} = \pareinv{\Xmatt\Vmat^{-1}\Xmat}\Xmatt\Vmat^{-1}\tvec 
\end{split}
$$


This results in an OLS solution that depends on the variance. Note that here the variance influences the OLS solution, in contrast to the multioutput model we studied before. This has some "consequences".

First, this connects with another viewpoint of this problem, which is known as generalized least squares, https://en.wikipedia.org/wiki/Generalized_least_squares, which arrives at this result through a different perspective. From what I know (which is not too much), in this approach one usually knows the $\Vmat$ rather than estimates it. This allows one to consider problems in which there is correlation between the points $n$, which makes $\Vmat$ a dense matrix.  From what I have understood, the idea is to remove this information in order to get a valid estimate for the OLS, which otherwise would be biased due to the influence of this heteroscedasticity and correlations.

However, this has two problems. First, knowing $\Vmat$, and then, this does not allow us to perform predictions at new $n$ considering this heteroscedasticity. How do we solve this?. Well, by making $\Vmat$ depend on the input in some way. Let's solve both problems.

### Estimating $\Vmat$.

What if $\Vmat$ is unknown and we want to learn it?. Well, we obviously can by a coordinate update on $\Vmat$. To do so, we need to obtain the gradient of the log-likelihood and set it to zero. The problem, however, is that we would need to estimate the variance at location $n$ from a single point $n$. This results in a variance of $0$, which does not make sense. 

How can we estimate the variance? Well, make it depend on the input through some sort of transformation. Since the variance is positive, we can use a different linear model than the one used to predict the mean. Since linear mappings can result in negative predictions, we can pass the output of this model through a transformation that always provides a positive value. This is achieved, for instance, by the exponential function. For a single point $n$, this is:

$$
z^n = {\xvect}^{(n)}\wvecsigma, \qquad \sigma_n^2 = e^{z^n}
$$

Stacking all $N$ points, in vector form:

$$
\zvec = \Xmat\wvecsigma, \qquad \Vvec = e^{\zvec}
$$

We can now feed this into our loss function, getting:

$$
\begin{split}
\log p(\tvec\mid\Xmat,\wvec,\Vvec) &= -\frac{N}{2}\log(2\pi) - \frac12\log\left|\Vvec\Imat\right| - \frac12\pareT{\tvec-\Xmat\wvec}\pareinv{\Vvec\Imat}\pare{\tvec-\Xmat\wvec}\\
&= \sum^N_{n=1}\pare{-\frac12\log(2\pi) - \frac12\log e^{{\xvect}^{(n)}\wvecsigma} - \frac{(t^n-{\xvect}^{(n)}\wvec)^2}{2e^{{\xvect}^{(n)}\wvecsigma}}}
\end{split}
$$

And now our goal is to maximize this likelihood wrt $\wvec$ and $\wvecsigma$:

$$
\begin{split}
&\argmax_{\wvec,\wvecsigma} -\frac{N}{2}\log(2\pi) - \frac12\log\left|\Vvec\Imat\right| - \frac12\pareT{\tvec-\Xmat\wvec}\pareinv{\Vvec\Imat}\pare{\tvec-\Xmat\wvec}\\
&=\argmin_{\wvec,\wvecsigma} \log\left|\Vvec\Imat\right| + \pareT{\tvec-\Xmat\wvec}\pareinv{\Vvec\Imat}\pare{\tvec-\Xmat\wvec}
\end{split}
$$


There are two ways in which this can be done. First, obtain gradients wrt $\wvec$ and $\wvecsigma$ and apply gradient descent. Another way, which should be faster, considers a coordinate descent approach where we can make exact gradient updates on $\wvec$ given $\wvecsigma$ and then a gradient descent step on $\wvecsigma$, since there is no closed-form update for this.

To see so, let's work out both steps.


#### Algorithm 1: Gradient descent.


We need the gradients wrt $\wvec$ and $\wvecsigma$. These can be obtained by applying the chain rule with the standard high school methods. No need for differentiation here, although in essence they do the same.

##### Gradient wrt $\wvec$

Let's start with $\nabla_\wvec$. Since $\wvecsigma$ is fixed for this gradient, $\Vvec=e^{\Xmat\wvecsigma}$ is a constant here, and the term $\log\left|\Vvec\Imat\right|$ does not depend on $\wvec$, so it drops out. Following the same style we used for the quadratic and absolute losses in the [regression theory notebook](1_Regression_Shallow_theory.ipynb), we express the $\wvec$-dependent part of the objective we are minimizing as a composition of vector functions. We are in the case $f:\mathbb{R}^D\to\mathbb{R}$, with $N$ training points: $\wvec\in\mathbb{R}^D$, $\Xmat\in\mathbb{R}^{N\times D}$ is the design matrix (not a vector), and $\tvec\in\mathbb{R}^N$. $\yvec,\mathbf{r}\in\mathbb{R}^N$; $\Vvec\in\mathbb{R}^N$ is held constant; $g$ is a scalar.

$$
\begin{align*}
\yvec &= \Xmat\wvec && \mathbb{R}^D \to \mathbb{R}^N\\
\mathbf{r} &= \tvec-\yvec && \mathbb{R}^N \to \mathbb{R}^N\\
L &= \mathbf{r}^T\pareinv{\Vvec\Imat}\mathbf{r} && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

where the Jacobians of each element of the composition are given by (using the differential of the quadratic form with the constant matrix $\pareinv{\Vvec\Imat}$ for the last stage, $\dd\bra{\mathbf{r}^T\pareinv{\Vvec\Imat}\mathbf{r}}=2\mathbf{r}^T\pareinv{\Vvec\Imat}\dd\mathbf{r}$):

$$
\begin{split}
J_{\mathbf{r}} &= 2\mathbf{r}^T\pareinv{\Vvec\Imat} \in \mathbb{R}^{1\times N}\\
J_\yvec &= -\Imat \in \mathbb{R}^{N\times N}\\
J_\wvec &= \Xmat \in \mathbb{R}^{N\times D}
\end{split}
$$

The total Jacobian  with respect to $\wvec$ is obtained by multiplying the Jacobians of each stage:

$$
\begin{split}
J_\wvec &= J_{\mathbf{r}}\,J_\yvec\,J_\wvec\\
&= \pare{2\mathbf{r}^T\pareinv{\Vvec\Imat}}\,\pare{-\Imat}\,\Xmat\\
&= -2\mathbf{r}^T\pareinv{\Vvec\Imat}\Xmat
\end{split}
$$

The gradient is the transposed Jacobian:

$$
\nabla_\wvec = -2\Xmatt\pareinv{\Vvec\Imat}\pare{\tvec-\Xmat\wvec}
$$


Other valid function compositions for this loss are:

$$
\begin{align*}
\yvec &= \Xmat\wvec && \mathbb{R}^D \to \mathbb{R}^N\\
\mathbf{r} &= \tvec-\yvec && \mathbb{R}^N \to \mathbb{R}^N\\
L &= \mathbf{r}^T\pare{\Vvec^{-1}\circ\mathbf{r}} && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

$$
\begin{align*}
\yvec &= \Xmat\wvec && \mathbb{R}^D \to \mathbb{R}^N\\
\mathbf{r} &= (\tvec-\yvec)^2 && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise function}\\
\mathbf{c} &= \Vvec^{-1}\circ\mathbf{r} && \mathbb{R}^N \to \mathbb{R}^N\\
L &= \onevect\mathbf{c} && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

**Exercise**: work out gradients and differentials to see we arrive at the same

##### Gradient wrt $\wvecsigma$


Here, the function composition is:

$$
\begin{split}
\zvec &= \Xmat\wvecsigma\\
\Vvec &= e^{\zvec}\\
L &= \log\left|\Vvec\Imat\right| + \avect\pareinv{\Vvec\Imat}\avec; \quad \avec = \tvec-\Xmat\wvec
\end{split}
$$

This differential can be obtained using the same tricks as in the "Independent per output variance" subsection above.

$$
\begin{split}
\log\left|\Vvec\Imat\right| = \log\left|\diag\pare{\Vvec}\right|
\end{split}
$$

Here, I am going to use a different path with new identities. Similar to before, the differential is given by:

$$
\begin{split}
&\vvec\braT{\pareinv{\diag\pare{\Vvec}}}R'\dd\Vvec = \\
& \vvec\braT{\diag\pare{\Vvec^{-1}}}R'\dd\Vvec\\
& \pareT{\Vvec^{-1}}RR'\dd\Vvec\\
& \pareT{\Vvec^{-1}}\dd\Vvec; \quad RR' = \Imat
\end{split}
$$

where we have applied eq 69 in Minka in the last step (this is something new to what we did before, and note that $RR'=\Imat$). Now, rather than identifying the quadratic form with a scalar, applying the trace, and then the cyclical property, we will directly apply the differential of the inverse and then all these properties from the trace. We have:

$$
\begin{split}
&\dd\bra{\avect\pareinv{\Vvec\Imat}\avec} = \\
&\dd\bra{\avect\pareinv{\diag\pare{\Vvec}}\avec} = \\
&-\avect\pareinv{\diag\pare{\Vvec}} \dd \bra{\diag\pare{\Vvec}}\pareinv{\diag\pare{\Vvec}}\avec = \\
&-\tr{\pareinv{\diag\pare{\Vvec}}\avec\avect\pareinv{\diag\pare{\Vvec}} \dd \bra{\diag\pare{\Vvec}}}\\
&-\vvec\braT{\pareinv{\diag\pare{\Vvec}}\avec\avect\pareinv{\diag\pare{\Vvec}}}\dd \vvec\bra{\diag\pare{\Vvec}}\\
&-\vvec\braT{\pareinv{\diag\pare{\Vvec}}\avec\avect\pareinv{\diag\pare{\Vvec}}}R'\dd \Vvec\\
&=-\vvec\braT{\Vvec^{-1}\pareinv{\Vvect}\circ\avec\avect}R'\dd \Vvec\\
&=-\braT{\diag^{-1}\pare{\Vvec^{-1}\pareinv{\Vvect}\circ\avec\avect}}\dd \Vvec\\
&=-\braT{\Vvec^{-1}\circ\Vvec^{-1}\circ\diag^{-1}\pare{\avec\avect}}\dd \Vvec\\
&=-\braT{\Vvec^{-1}\circ\Vvec^{-1}\circ\avec\circ\avec}\dd \Vvec
\end{split}
$$

So the Jacobians are:

$$
\begin{split}
J_\Vvec = \pareT{\Vvec^{-1}} -\braT{\Vvec^{-1}\circ\Vvec^{-1}\circ\avec\circ\avec}\\
J_\zvec = \diag(e^{\zvec})\\
J_{\wvecsigma} = \Xmat\\
\end{split}
$$

The gradient is:

$$
\nabla_{\wvecsigma} = \Xmatt\diag(e^{\Xmat\wvecsigma})\pare{\pareinv{e^{\Xmat\wvecsigma}} -\pareinv{e^{\Xmat\wvecsigma}}\circ\pareinv{e^{\Xmat\wvecsigma}}\circ\pare{\tvec-\Xmat\wvec}\circ\pare{\tvec-\Xmat\wvec}}
$$

With these two gradients, we can perform gradient descent on the parameters $\wvec$ and $\wvecsigma$.

#### Algorithm 2 Coordinate optimization: exact and gradient-based coordinate updates

For this particular problem, we can derive a better algorithm that converges in smaller steps. Note that gradient descent only uses first-order information on each step. On the other side, coordinate descent can do full steps, exploiting the geometry of the problem. It turns out that, in this problem, we can perform a coordinate update on $\wvec$ and a gradient descent update on $\wvecsigma$. The coordinate update on $\wvec$ has already been derived and is given by:

$$
\begin{split}
&\wvec_\text{opt} = \pareinv{\Xmatt\diag\pareinv{e^{\Xmat\wvecsigma}}\Xmat}\Xmatt\diag\pareinv{e^{\Xmat\wvecsigma}}\tvec 
\end{split}
$$

So the algorithm alternates between the following two steps:

**Step 1** ($\wvec$, exact): given $\wvecsigma^{(t)}$, update

$$
\wvec_\text{opt}^{(t+1)} = \pareinv{\Xmatt\diag\pareinv{e^{\Xmat\wvecsigma^{(t)}}}\Xmat}\Xmatt\diag\pareinv{e^{\Xmat\wvecsigma^{(t)}}}\tvec
$$

**Step 2** ($\wvecsigma$, gradient step): given $\wvec_\text{opt}^{(t+1)}$ from Step 1, update

$$
\wvecsigma^{(t+1)} = \wvecsigma^{(t)} - \eta\,\Xmatt\diag\pare{e^{\Xmat\wvecsigma^{(t)}}}\pare{\pareinv{e^{\Xmat\wvecsigma^{(t)}}} - \pareinv{e^{\Xmat\wvecsigma^{(t)}}}\circ\pareinv{e^{\Xmat\wvecsigma^{(t)}}}\circ\avec^{(t+1)}\circ\avec^{(t+1)}}, \qquad \avec^{(t+1)}=\tvec-\Xmat\wvec_\text{opt}^{(t+1)}
$$

Note that $\wvec_\text{opt}^{(t+1)}$ (just updated in Step 1) is what enters $\avec^{(t+1)}$ in Step 2, while $\wvecsigma$ itself is still $\wvecsigma^{(t)}$ everywhere else in Step 2 — it is only updated at the very end, producing $\wvecsigma^{(t+1)}$.

### Experiment

Let's fit this model to our data.

#### Normalization

Before fitting anything, we standardize (z-score) both $\xvec$ (temperature) and $\tvec$ (sales). This matters more here than in the homoscedastic case: with the exponential link $\Vvec=e^{\Xmat\wvecsigma}$, even a moderate step on $\wvecsigma$ can make $e^{\Xmat\wvecsigma}$ overflow if $\Xmat$ is not on a roughly unit scale. Below, before vs. after. Also, since the exponential function has a drastic change in curvature, gradient descent can be very sensitive to the learning rate.

In [ ]:
temperature_z = (temperature_hetero - temperature_hetero.mean()) / temperature_hetero.std()
sales_z = (sales_hetero - sales_hetero.mean()) / sales_hetero.std()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(temperature_hetero, sales_hetero, marker="x")
axes[0].set_xlabel("temperature")
axes[0].set_ylabel("sales")
axes[0].set_title("Before normalization")
axes[0].grid(True)

axes[1].scatter(temperature_z, sales_z, marker="x")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (z-score)")
axes[1].set_title("After normalization")
axes[1].grid(True)

plt.show()

#### Comparing convergence speed: Algorithm 1 (gradient descent) vs. Algorithm 2 (coordinate optimization)

As we did for the Student-t model, we compare how fast each algorithm converges. We try two initializations for $(\wvec,\wvecsigma)$: the OLS solution for $\wvec$ together with $\wvecsigma=0$ (ie constant variance to start with), and a random one. That gives 4 runs in total (2 algorithms x 2 initializations).

In [ ]:
x_data = temperature_z.reshape(-1, 1)
t_data = sales_z.reshape(-1, 1)

lr = 1e-4
epochs = 500

# initialization 1: OLS solution for w, constant variance (w_sigma=0) to start with
w0_ols, b0_ols = create_computation_graph_linear_from_ols(x_data, t_data)
w0_sigma_ols, b0_sigma_ols = np.zeros((1,1)), np.zeros((1,1))

# initialization 2: random
np.random.seed(0)
w0_rand, b0_rand = create_computation_graph_linear(1, 1)
w0_sigma_rand, b0_sigma_rand = create_computation_graph_linear(1, 1)

def run_gradient_descent(w, b, w_sigma, b_sigma, epochs=epochs, lr=lr):
    w, b, w_sigma, b_sigma = w.copy(), b.copy(), w_sigma.copy(), b_sigma.copy()
    loss_history = []
    for e in range(epochs):
        y_pred, v_pred = computation_graph_heteroscedastic_gaussian(x_data, w, b, w_sigma, b_sigma)
        loss_history.append(np.sum(heteroscedastic_loss_function(t_data, y_pred, v_pred)))

        grad_w, grad_b, grad_w_sigma, grad_b_sigma = grad_heteroscedastic_loss_wrt_linear_model(
            x_data, t_data, w, b, w_sigma, b_sigma)
        w = w - lr * grad_w
        b = b - lr * grad_b
        w_sigma = w_sigma - lr * grad_w_sigma
        b_sigma = b_sigma - lr * grad_b_sigma

    return loss_history

def run_coordinate_descent(w, b, w_sigma, b_sigma, epochs=epochs, lr=lr):
    w, b, w_sigma, b_sigma = w.copy(), b.copy(), w_sigma.copy(), b_sigma.copy()
    loss_history = []
    for e in range(epochs):
        y_pred, v_pred = computation_graph_heteroscedastic_gaussian(x_data, w, b, w_sigma, b_sigma)
        loss_history.append(np.sum(heteroscedastic_loss_function(t_data, y_pred, v_pred)))

        # Step 1 (w, exact), given the current w_sigma
        w, b = fit_ols_heteroscedastic(x_data, t_data, w_sigma, b_sigma)

        # Step 2 (w_sigma, gradient step), given the just-updated w
        grad_w_sigma, grad_b_sigma = grad_heteroscedastic_loss_wrt_linear_model(
            x_data, t_data, w, b, w_sigma, b_sigma, only_sigma=True)
        w_sigma = w_sigma - lr * grad_w_sigma
        b_sigma = b_sigma - lr * grad_b_sigma

    return loss_history

gd_ols_loss = run_gradient_descent(w0_ols, b0_ols, w0_sigma_ols, b0_sigma_ols)
gd_rand_loss = run_gradient_descent(w0_rand, b0_rand, w0_sigma_rand, b0_sigma_rand)
cd_ols_loss = run_coordinate_descent(w0_ols, b0_ols, w0_sigma_ols, b0_sigma_ols)
cd_rand_loss = run_coordinate_descent(w0_rand, b0_rand, w0_sigma_rand, b0_sigma_rand)

plt.figure(figsize=(7, 5))
plt.plot(gd_ols_loss, label="Algorithm 1 (GD), OLS init")
plt.plot(gd_rand_loss, label="Algorithm 1 (GD), random init")
plt.plot(cd_ols_loss, label="Algorithm 2 (coord.), OLS init")
plt.plot(cd_rand_loss, label="Algorithm 2 (coord.), random init")
plt.xlabel("Epoch")
plt.ylabel("Loss (heteroscedastic)")
plt.title("Convergence speed: Algorithm 1 vs Algorithm 2, OLS vs random init")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# fit the mean and variance models to convergence (coordinate descent, OLS init)
w_plot, b_plot, w_sigma_plot, b_sigma_plot = w0_ols.copy(), b0_ols.copy(), w0_sigma_ols.copy(), b0_sigma_ols.copy()
for e in range(2000):
    w_plot, b_plot = fit_ols_heteroscedastic(x_data, t_data, w_sigma_plot, b_sigma_plot)
    grad_w_sigma, grad_b_sigma = grad_heteroscedastic_loss_wrt_linear_model(
        x_data, t_data, w_plot, b_plot, w_sigma_plot, b_sigma_plot, only_sigma=True)
    w_sigma_plot = w_sigma_plot - lr * grad_w_sigma
    b_sigma_plot = b_sigma_plot - lr * grad_b_sigma

x_line = np.linspace(x_data.min(), x_data.max(), 100).reshape(-1, 1)
y_line = x_line @ w_plot + b_plot
sigma_line = np.sqrt(computation_graph_heteroscedastic_variance(x_line, w_sigma_plot, b_sigma_plot))

plt.figure(figsize=(7, 5))
plt.scatter(x_data, t_data, marker="x", label="data")
plt.plot(x_line, y_line, color="red", label="fitted mean")
plt.fill_between(x_line.ravel(), (y_line - sigma_line).ravel(), (y_line + sigma_line).ravel(),
                  color="red", alpha=0.2, label="±1 std (heteroscedastic)")
plt.xlabel("temperature (z-score)")
plt.ylabel("sales (z-score)")
plt.title("Fitted mean and heteroscedastic uncertainty")
plt.legend()
plt.grid(True)
plt.show()

### Non-linear $\Vmat$.

In the previous experiment, we see that the variance model cannot correctly capture the shape of the variance. The variance expands first at values around $-1.5$, shrinks at $-0.5$, and expands again. Why, well, because we have considered a linear relation $\xvect\wvecsigma$. This can only capture linear increase or decrease. Then, since it is passed through a monotone function (link function $e^{x}$, which does not change the order even if it is non-linear), the order is preserved, and thus we can only learn a variance that increases or decreases.

How can we model this relationship? Well, in a similar way to what we did in the linear basis function models. There, we were using linear basis functions to represent the mean. Here we can do the same to learn non-linear variances. For instance, we can replace $\xvec$ by a vector of polynomial basis functions of $\xvec$ before feeding it into $\wvecsigma$. Let's use a polynomial of order 4:

$$
\phi\pare{\xvec} = \pare{\xvec,\xvec^2,\xvec^3,\xvec^4}
$$

and define $z$, and thus the variance, as:

$$
z^n = \phi\pare{\xvec^{(n)}}^T\wvecsigma + b_\sigma, \qquad \sigma_n^2 = e^{z^n}
$$

Now $\wvecsigma\in\mathbb{R}^4$, one weight per power of $\xvec$, instead of $\wvecsigma\in\mathbb{R}$. Since $z^n$ is a degree-4 polynomial in $\xvec^{(n)}$ instead of a linear function of it, it is no longer forced to be monotone: it can decrease, then increase, then decrease again, so $\sigma_n^2=e^{z^n}$ can now shrink and expand as many times as a degree-4 polynomial allows, instead of just once. Everything else: the loss, the gradients, and the coordinate update for $\wvec$, stays exactly the same as before: we are just feeding $\phi(\xvec)$ instead of $\xvec$ into the same linear-then-exponential model for the variance.

#### Experiment

We fit this non-linear-variance model with the same coordinate algorithm as before (Algorithm 2): an exact weighted least squares update for $\wvec$ given the current $\wvecsigma$, and a gradient step on $\wvecsigma$ given the just-updated $\wvec$. We start from the OLS solution for $\wvec$, with $\wvecsigma=0$ (constant variance) to begin with.

In [ ]:
# degree-4 polynomial features for the variance model, mean model keeps x_data as-is
phi_data = generate_poly_features(x_data, poly_degree=4, add_bias=False)

# coordinate descent, starting from the OLS solution for w, w_sigma=0 (constant variance)
w_poly, b_poly = w0_ols.copy(), b0_ols.copy()
w_sigma_poly, b_sigma_poly = np.zeros((4, 1)), np.zeros((1, 1))

epochs_poly = 3000
loss_history_poly = []

for e in range(epochs_poly):
    y_pred, v_pred = computation_graph_heteroscedastic_gaussian(
        x_data, w_poly, b_poly, w_sigma_poly, b_sigma_poly, x_sigma=phi_data)
    loss_history_poly.append(np.sum(heteroscedastic_loss_function(t_data, y_pred, v_pred)))

    # Step 1 (w, exact): weighted least squares given the current w_sigma
    w_poly, b_poly = fit_ols_heteroscedastic(x_data, t_data, w_sigma_poly, b_sigma_poly, x_sigma=phi_data)

    # Step 2 (w_sigma, gradient step): given the just-updated w
    grad_w_sigma, grad_b_sigma = grad_heteroscedastic_loss_wrt_linear_model(
        x_data, t_data, w_poly, b_poly, w_sigma_poly, b_sigma_poly, only_sigma=True, x_sigma=phi_data)
    w_sigma_poly = w_sigma_poly - lr * grad_w_sigma
    b_sigma_poly = b_sigma_poly - lr * grad_b_sigma

# resulting mean and heteroscedastic variance
x_line = np.linspace(x_data.min(), x_data.max(), 100).reshape(-1, 1)
phi_line = generate_poly_features(x_line, poly_degree=4, add_bias=False)
y_line_poly = x_line @ w_poly + b_poly
sigma_line_poly = np.sqrt(computation_graph_heteroscedastic_variance(phi_line, w_sigma_poly, b_sigma_poly))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(loss_history_poly)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (heteroscedastic)")
axes[0].set_title("Convergence: coordinate descent with degree-4 polynomial variance")
axes[0].grid(True)

axes[1].scatter(x_data, t_data, marker="x", label="data")
axes[1].plot(x_line, y_line_poly, color="red", label="fitted mean")
axes[1].fill_between(x_line.ravel(), (y_line_poly - sigma_line_poly).ravel(), (y_line_poly + sigma_line_poly).ravel(),
                      color="red", alpha=0.2, label="±1 std (heteroscedastic)")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (z-score)")
axes[1].set_title("Fitted mean and heteroscedastic uncertainty")
axes[1].legend()
axes[1].grid(True)

### Estimating dense $\Vmat$.

Estimating a dense matrix would allow us to consider correlation between points. Here the problem, however, is that our function needs to output a valid correlation matrix between any pair of points, so that the resulting matrix $\Vmat$ is positive definite. At this moment I am not aware of how we can do this through what we have seen here. Here, probably, the path is to directly go into Gaussian Processes to figure out if our $\sigma_n$ function can be directly obtained from a kernel.

## Homoscedastic Laplace Likelihood

The density of the Laplace distribution (with location $\mu$ and scale $\sigma$) is:

$$
p(x \mid \mu, \sigma) = \frac{1}{2\sigma}\exp\left(-\frac{|x-\mu|}{\sigma}\right)
$$

Compared to the Gaussian, it has a sharper peak at $\mu$ and heavier tails (though lighter than the Student-t's), which makes it more robust to outliers, as we will see.

### Likelihood for $N$ training points

Assuming each observation $t^{(n)}$ is generated from the linear model $y^{(n)} = {\xvect}^{(n)}\wvec$ (with $\wvec$ absorbing the bias and ${\xvect}^{(n)}$ the extended input vector of point $n$) with Laplace noise of scale $\sigma$, and that the observations are independent of each other given the model, the joint likelihood for $N$ points $\{(x^{(n)},t^{(n)})\}_{n=1}^N$ is:

$$
p(\tvec\mid \Xmat, \wvec, \sigma) = \prod_{n=1}^N p(t^{(n)} \mid {\xvect}^{(n)}, \wvec, \sigma) = \prod_{n=1}^N \frac{1}{2\sigma}\exp\left(-\frac{|t^{(n)} - {\xvect}^{(n)}\wvec|}{\sigma}\right)
$$

Taking the logarithm and expanding each term:

$$
\begin{split}
\log p(\tvec\mid {\xvect}^{(n)}, \wvec, \sigma) &= \log \prod_{n=1}^N p(t^{(n)} \mid {\xvect}^{(n)}, \wvec, \sigma) \\
&= \sum_{n=1}^N \log p(t^{(n)} \mid {\xvect}^{(n)}, \wvec, \sigma) \\
&= \sum_{n=1}^N \log\left[\frac{1}{2\sigma}\exp\left(-\frac{|t^{(n)} - {\xvect}^{(n)}\wvec|}{\sigma}\right)\right] \\
&= \sum_{n=1}^N \left[-\log(2\sigma) - \frac{|t^{(n)} - {\xvect}^{(n)}\wvec|}{\sigma}\right] \\
&= -N\log(2\sigma) - \frac{1}{\sigma}\sum_{n=1}^N |t^{(n)} - {\xvect}^{(n)}\wvec|
\end{split}
$$

Maximizing the log-likelihood is equivalent to minimizing its negative. The term $N\log(2\sigma)$ does not depend on $\tvec,{\xvect}^{(n)},\wvec$, and the factor $\frac{1}{\sigma}$ is a positive constant with respect to the optimization over $\wvec$. Using only the terms that depend on $\tvec,{\xvect}^{(n)},\wvec$, the resulting cost function to minimize is:

$$
L(\wvec) = \sum_{n=1}^N |t^{(n)} - {\xvect}^{(n)}\wvec|
$$

which is exactly the absolute loss (L1) that we already used as a loss function. In this case, optimization can only be done through gradient descent.

### Optimal estimate of $\sigma$

We differentiate the log-likelihood with respect to $\sigma$ and set it to zero:

$$
\begin{split}
\frac{\partial}{\partial \sigma}\left[-N\log(2\sigma) - \frac{1}{\sigma}\sum_{n=1}^N |t^{(n)} - {\xvect}^{(n)}\wvec|\right] &= -\frac{N}{\sigma} + \frac{1}{\sigma^2}\sum_{n=1}^N |t^{(n)} - {\xvect}^{(n)}\wvec| = 0\\
\sigma_{\text{opt}} &= \frac{1}{N}\sum_{n=1}^N |t^{(n)} - {\xvect}^{(n)}\wvec|
\end{split}
$$

As we saw before, $\wvec_{\text{opt}}$ (the one that minimizes $L(\wvec)=\sum_n|t^{(n)}-{\xvect}^{(n)}\wvec|$) does not depend on $\sigma$, since $\frac{1}{\sigma}$ is a global multiplicative constant that does not affect the location of the minimum. So no iterative scheme is needed: it suffices to first obtain $\wvec_{\text{opt}}$ (independently of $\sigma$) and substitute it directly:

$$
\sigma_{\text{opt}} = \frac{1}{N}\sum_{n=1}^N \left|t^{(n)} - {\xvect}^{(n)}\wvec_{\text{opt}}\right|
$$ 

Again, this results in a coordinate descent algorithm where we first find $\wvec_{\text{opt}}$ through gradient descent, and then $\sigma$ by running the above computation.

## Homoscedastic Student-t Likelihood

The density of the Student-t distribution (with location $\mu$, scale $\sigma^2$, and $\nu$ degrees of freedom) is:

$$
p(x \mid \mu, \sigma^2, \nu) = \frac{\Gamma\left(\frac{\nu+1}{2}\right)}{\Gamma\left(\frac{\nu}{2}\right)\sqrt{\nu\pi\sigma^2}}\left(1 + \frac{(x-\mu)^2}{\nu\sigma^2}\right)^{-\frac{\nu+1}{2}}
$$

where $\Gamma(\cdot)$ is the Gamma function. As $\nu \to \infty$, this density converges to a Gaussian $\mathcal{N}(\mu,\sigma^2)$; for small values of $\nu$, the tails are much heavier than the Gaussian's, which makes it more robust to outliers. We parametrize the density directly by $\sigma^2$ (rather than $\sigma$) so it matches the Gaussian's own variance parameter, and so it stays consistent with every derivation from here on, which will only ever need $\sigma^2$.

### Likelihood for $N$ training points

Assuming each observation $t^{(n)}$ is generated from the linear model $y^{(n)} = {\xvect}^{(n)}\wvec$ (with $\wvec$ absorbing the bias and ${\xvect}^{(n)}$ the extended input vector of point $n$) with Student-t noise of scale $\sigma^2$ and $\nu$ degrees of freedom, and that the observations are independent of each other given the model, the joint likelihood for $N$ points $\{(x^{(n)},t^{(n)})\}_{n=1}^N$ is:

$$
p(\tvec\mid \Xmat, \wvec, \sigma^2, \nu) = \prod_{n=1}^N p(t^{(n)} \mid {\xvect}^{(n)}, \wvec, \sigma^2, \nu) = \prod_{n=1}^N \frac{\Gamma\left(\frac{\nu+1}{2}\right)}{\Gamma\left(\frac{\nu}{2}\right)\sqrt{\nu\pi\sigma^2}}\left(1 + \frac{(t^{(n)} - {\xvect}^{(n)}\wvec)^2}{\nu\sigma^2}\right)^{-\frac{\nu+1}{2}}
$$

Taking the logarithm and expanding each term:

$$
\begin{split}
\log p(\tvec\mid {\xvect}^{(n)}, \wvec, \sigma^2, \nu) &= \log \prod_{n=1}^N p(t^{(n)} \mid {\xvect}^{(n)}, \wvec, \sigma^2, \nu) \\
&= \sum_{n=1}^N \log p(t^{(n)} \mid {\xvect}^{(n)}, \wvec, \sigma^2, \nu) \\
&= \sum_{n=1}^N \log\left[\frac{\Gamma\left(\frac{\nu+1}{2}\right)}{\Gamma\left(\frac{\nu}{2}\right)\sqrt{\nu\pi\sigma^2}}\left(1 + \frac{(t^{(n)} - {\xvect}^{(n)}\wvec)^2}{\nu\sigma^2}\right)^{-\frac{\nu+1}{2}}\right] \\
&= \sum_{n=1}^N \left[\log\Gamma\left(\frac{\nu+1}{2}\right) - \log\Gamma\left(\frac{\nu}{2}\right) - \frac{1}{2}\log(\nu\pi\sigma^2) - \frac{\nu+1}{2}\log\left(1 + \frac{(t^{(n)} - {\xvect}^{(n)}\wvec)^2}{\nu\sigma^2}\right)\right] \\
&= N\log\Gamma\left(\frac{\nu+1}{2}\right) - N\log\Gamma\left(\frac{\nu}{2}\right) - \frac{N}{2}\log(\nu\pi\sigma^2) - \frac{\nu+1}{2}\sum_{n=1}^N\log\left(1 + \frac{(t^{(n)} - {\xvect}^{(n)}\wvec)^2}{\nu\sigma^2}\right)
\end{split}
$$

Maximizing the log-likelihood is equivalent to minimizing its negative. The terms $N\log\Gamma\left(\frac{\nu+1}{2}\right)$, $N\log\Gamma\left(\frac{\nu}{2}\right)$, $\frac{N}{2}\log(\nu\pi\sigma^2)$, and the factor $\frac{\nu+1}{2}$ do not depend on $\tvec,{\xvect}^{(n)},\wvec$, so they are constants with respect to the optimization over $\wvec$. Using only the terms that depend on $\tvec,{\xvect}^{(n)},\wvec$, the resulting cost function to minimize is:

$$
L(\wvec) = \sum_{n=1}^N \log\left(1 + \frac{(t^{(n)} - {\xvect}^{(n)}\wvec)^2}{\nu\sigma^2}\right)
$$

This is a new loss function we have not seen yet but is derived from the Student-t distribution. Since we know properties from this distribution, we will see that optimizing this loss results in a model that inherits them. In this case, since the Student-t is heavy-tailed, this makes the likelihood assign higher values to points far from the mean. Consequently, if we assume our data follows a Student-t distribution with mean given by our model, this extreme data will not affect the fitting process as much as the Gaussian likelihood. We will see this later on. Let's first derive algorithms to fit this model.



### Optimization through gradient descent

While we will see that the best we can do to fit this model is using advanced versions of the EM algorithm (which is a coordinate descent method). Let's start studying the naive approach through gradient descent.

#### Optimizing the loss wrt $\wvec$

Since this is a new loss, let's study how we can optimize it.


##### The cost as a composition of functions

Just as we did with the squared and absolute losses in the [regression theory](1_Regression_Shallow_theory.ipynb), we can express $L(\wvec)$ as a composition of vector-valued functions. We are in the case $f:\mathbb{R}^D\to\mathbb{R}$ (scalar-output model), with $N$ training points: $\wvec\in\mathbb{R}^{D+1}$ is a vector (bias absorbed), $\Xmat\in\mathbb{R}^{N\times(D+1)}$ is the design matrix, and $\tvec\in\mathbb{R}^N$, $\onevec\in\mathbb{R}^N$. The remaining elements of the composition, $\yvec,\mathbf{r},\cvec$, are vectors in $\mathbb{R}^N$ (one per training point); $\nu,\sigma^2$ are scalars, and $L$ is a scalar.

$$
\begin{align*}
\yvec &= \Xmat\wvec && \mathbb{R}^{D+1} \to \mathbb{R}^N\\
\mathbf{r} &= (\tvec-\yvec)^2 && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise function}\\
\cvec &= \log\left(\onevec + \frac{1}{\nu\sigma^2}\mathbf{r}\right) && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise function}\\
L &= \onevect\cvec && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

where the Jacobians of each element of the composition are given by:

$$
\begin{split}
J_\cvec &= \onevect \in \mathbb{R}^{1\times N}\\
J_{\mathbf{r}} &= \diag\left(\frac{1}{\nu\sigma^2 + \mathbf{r}}\right) \in \mathbb{R}^{N \times N}\\
J_\yvec &= -2\diag(\tvec-\yvec) \in \mathbb{R}^{N \times N}\\
J_\wvec &= \Xmat \in \mathbb{R}^{N \times (D+1)}
\end{split}
$$

The total Jacobian of $L$ with respect to $\wvec$ is obtained by multiplying the Jacobians of each stage:

$$
\begin{split}
J_\wvec &= J_\cvec\, J_{\mathbf{r}}\, J_\yvec\, J_\wvec\\
&= -2\,\onevect\,\diag\left(\frac{1}{\nu\sigma^2 + \mathbf{r}}\right)\diag(\tvec-\yvec)\,\Xmat
\end{split}
$$

The gradient is the transpose of the Jacobian:

$$
\begin{split}
\nabla_\wvec &= J_\wvec^T\\
&= -2\,\Xmatt\,\diag(\tvec-\yvec)\,\diag\left(\frac{1}{\nu\sigma^2 + \mathbf{r}}\right)\,\onevec
\end{split}
$$

Substituting $\yvec=\Xmat\wvec$ and $\mathbf{r}=(\tvec-\yvec)^2=(\tvec-\Xmat\wvec)^2$, everything is left in terms of $\tvec,\Xmat,\wvec,\nu,\sigma^2$:

$$
\begin{split}
\nabla_\wvec &= -2\,\Xmatt\,\diag(\tvec-\Xmat\wvec)\,\diag\left(\frac{1}{\nu\sigma^2 + (\tvec-\Xmat\wvec)^2}\right)\,\onevec\\
&= -2\sum_{n=1}^N {\xvec}^{(n)}\,\frac{t^{(n)}-{\xvect}^{(n)}\wvec}{\nu\sigma^2+(t^{(n)}-{\xvect}^{(n)}\wvec)^2}
\end{split}
$$


We can use this gradient in a gradient descent routine.


#### Optimizing the loss wrt $\sigma^2_{\text{opt}}$


The previous section shows that the gradient wrt $\wvec$ depends on the parameter $\sigma^2$. There is no way to isolate it from this gradient. This means that the optimal value for $\wvec$ cannot be obtained independently of the value of $\sigma^2$. In other words, we need to solve the minimum for those parameters at the same time. It is not like the Laplace likelihood where we can run gradient descent up to convergence, and then obtain a closed-form coordinate update for $\sigma$.

So optimization here will run by running gradient descent on both parameters at the same time. Starting from the expression already obtained:

$$
\log p(\tvec\mid \Xmat, \wvec, \sigma^2, \nu) = N\log\Gamma\left(\frac{\nu+1}{2}\right) - N\log\Gamma\left(\frac{\nu}{2}\right) - \frac{N}{2}\log(\nu\pi\sigma^2) - \frac{\nu+1}{2}\sum_{n=1}^N\log\left(1 + \frac{r_n^2}{\nu\sigma^2}\right)
$$

with $r_n = t^{(n)}-{\xvect}^{(n)}\wvec$, we differentiate with respect to $\sigma^2$ (the first two terms do not depend on $\sigma^2$):

$$
\begin{split}
\frac{\partial}{\partial\sigma^2}\log p(\tvec\mid \Xmat, \wvec, \sigma^2, \nu) &= -\frac{N}{2\sigma^2} - \frac{\nu+1}{2}\sum_{n=1}^N \frac{\partial}{\partial\sigma^2}\log\left(1+\frac{r_n^2}{\nu\sigma^2}\right)\\
&= -\frac{N}{2\sigma^2} - \frac{\nu+1}{2}\sum_{n=1}^N \frac{1}{1+\frac{r_n^2}{\nu\sigma^2}}\left(-\frac{r_n^2}{\nu(\sigma^2)^2}\right)\\
&= -\frac{N}{2\sigma^2} - \frac{\nu+1}{2}\sum_{n=1}^N \frac{-r_n^2}{\left(1+\frac{r_n^2}{\nu\sigma^2}\right)\nu(\sigma^2)^2}\\
&= -\frac{N}{2\sigma^2} - \frac{\nu+1}{2}\sum_{n=1}^N \frac{-r_n^2}{\nu(\sigma^2)^2+r_n^2\sigma^2}\\
&= -\frac{N}{2\sigma^2} + \frac{\nu+1}{2}\sum_{n=1}^N \frac{r_n^2}{\sigma^2(\nu\sigma^2+r_n^2)}\\
\end{split}
$$

We might want to try to see if there is a closed-form solution for this subproblem by setting the gradient to zero. There is no **Exercise**. This implies that we cannot use a hybrid optimization algorithm where we optimize $\wvec$ through a step of gradient descent and $\sigma^2$ through a coordinate update. Thus, we can use the gradient wrt this parameter to perform updates based on gradient descent. Here we have not expressed the cost as a composition of functions, but one can do it and check the result holds **Exercise**.


#### Optimizing the loss wrt $\nu$

The last parameter to update is the degrees of freedom. Again, there will no be closed form update of this parameter. Thus, we need to obtain the gradient to run gradient descent.

The digamma function $\psi(x)$ is defined as the derivative of the log-gamma function: $\psi(x) := \frac{d}{dx}\log\Gamma(x)$. We will use it to differentiate the two Gamma terms below.

Starting from the log likelihood function, we have:

$$
\log p(\tvec\mid \Xmat, \wvec, \sigma^2, \nu) = N\log\Gamma\left(\frac{\nu+1}{2}\right) - N\log\Gamma\left(\frac{\nu}{2}\right) - \frac{N}{2}\log(\nu\pi\sigma^2) - \frac{\nu+1}{2}\sum_{n=1}^N\log\left(1 + \frac{r_n^2}{\nu\sigma^2}\right)
$$

with $r_n=t^{(n)}-{\xvect}^{(n)}\wvec$. Differentiating each term with respect to $\nu$:

$$
\frac{\partial}{\partial\nu}\log p(\tvec\mid \Xmat,\wvec,\sigma^2,\nu) = \frac{N}{2}\psi\!\left(\frac{\nu+1}{2}\right) - \frac{N}{2}\psi\!\left(\frac{\nu}{2}\right) - \frac{N}{2\nu} - \frac12\sum_{n=1}^N\log\!\left(1+\frac{r_n^2}{\nu\sigma^2}\right) - \frac{\nu+1}{2}\sum_{n=1}^N\frac{\partial}{\partial\nu}\log\!\left(1+\frac{r_n^2}{\nu\sigma^2}\right)
$$

where

$$
\frac{\partial}{\partial\nu}\log\left(1+\frac{r_n^2}{\nu\sigma^2}\right) = \frac{1}{1+\frac{r_n^2}{\nu\sigma^2}}\left(-\frac{r_n^2}{\nu^2\sigma^2}\right) = -\frac{r_n^2}{\nu(\nu\sigma^2+r_n^2)}
$$

Substituting back:

$$
\frac{\partial}{\partial\nu}\log p(\tvec\mid \Xmat,\wvec,\sigma^2,\nu) = \frac{N}{2}\psi\!\left(\frac{\nu+1}{2}\right) - \frac{N}{2}\psi\!\left(\frac{\nu}{2}\right) - \frac{N}{2\nu} - \frac12\sum_{n=1}^N\log\!\left(1+\frac{\pare{t^{(n)}-{\xvect}^{(n)}\wvec}^2}{\nu\sigma^2}\right) + \frac{\nu+1}{2\nu}\sum_{n=1}^N\frac{\pare{t^{(n)}-{\xvect}^{(n)}\wvec}^2}{\nu\sigma^2+\pare{t^{(n)}-{\xvect}^{(n)}\wvec}^2}
$$



### Optimization through the EM algorithm (coordinate descent)

It turns out that improved convergence speed can be achieved through a modified version of the EM algorithm. The full description and comparison of the different variants is given in https://www3.stat.sinica.edu.tw/statistica/oldpdf/a5n12.pdf. Here we will do some minor derivations, writing down the most efficient version of this algorithm. A possible exercise would include implementation and comparison of the different variants described in the paper.


#### Optimizing wrt $\wvec$

Let's take a look at the gradient obtained in the step before:

$$
\begin{split}
\nabla_\wvec &= -2\,\Xmatt\,\diag(\tvec-\Xmat\wvec)\,\diag\left(\frac{1}{\nu\sigma^2 + (\tvec-\Xmat\wvec)^2}\right)\,\onevec\\
&= -2\sum_{n=1}^N {\xvec}^{(n)}\,\frac{t^{(n)}-{\xvect}^{(n)}\wvec}{\nu\sigma^2+(t^{(n)}-{\xvect}^{(n)}\wvec)^2}
\end{split}
$$

We observe that this gradient is pretty similar to the gradient of the squared loss, in the heteroscedastic setting with diagonal $\Vmat$. If we write $\pi_n = \frac{1}{\nu\sigma^2+(t^{(n)}-{\xvect}^{(n)}\wvec)^2}$, then

$$
\begin{split}
\nabla_\wvec &=-2\sum_{n=1}^N \pi_n {\xvec}^{(n)} \pare{t^{(n)}-{\xvect}^{(n)}\wvec}
\end{split}
$$

This means that if we let $\Pi$ denote the diagonal matrix with entries made up of $\pi_n$, this gradient can be rewritten as $-2\Xmatt\Pi\pare{\tvec-\Xmat\wvec}$. Setting the gradient to zero yields:

$$
\begin{split}
&\nabla_\wvec = 0\\
&-2\Xmatt\Pi\pare{\tvec-\Xmat\wvec} = 0\\
&\Xmatt\Pi\tvec=\Xmatt\Pi\Xmat\wvec\\
&\wvec_\text{opt} = \pareinv{\Xmatt\Pi\Xmat}\Xmatt\Pi\tvec 
\end{split}
$$

However, since the optimal $\wvec$ depends on $\Pi$, which in turns depends on $\wvec$ through $\nu\sigma^2+(t^{(n)}-{\xvect}^{(n)}\wvec)^2$ this means that the $\wvec_\text{opt}$  is not a global minimizer, but a minimizer of the suproblem obtained by fixing $\Pi$. Thus, the following iterative algorithm (which is a coordinate descent algorithm) can be used to obtain the minimum:

1. With $\wvec^{(t)}$, compute $r_n^{(t)}=t^{(n)}-{\xvect}^{(n)}\wvec^{(t)}$ and $\pi_n^{(t)}=\dfrac{1}{\nu\sigma^2+(r_n^{(t)})^2}$.
2. Update $\wvec^{(t+1)} = \pareinv{\Xmatt\mathbf{\Pi}^{(t)}\Xmat}\Xmatt\mathbf{\Pi}^{(t)}\tvec$.
3. Repeat until convergence.

This is a special case of the reweighted least squares method: https://en.wikipedia.org/wiki/Iteratively_reweighted_least_squares. The question is, how do we ensure this actually converges? Well, it turns out that it is a type of majorize-minimization algorithm. I do not know much about this. However, again, the probabilistic reinterpretation of this problem provides an answer to this. 

It turns out that if we use the Gaussian scale-mixture representation of the Student-t, we can treat this problem as a latent variable model and use the expectation-maximization algorithm to fit the problem. Part of the E-step is exactly computing the weight we have computed. The M-step for $\wvec$ is exactly the least-squares solution. So we can see step 1 of the algorithm as the E-step and step 2 as the M-step. Further information here https://www3.stat.sinica.edu.tw/statistica/oldpdf/a5n12.pdf.

In reality, the E-step results in a $\pi_n$ given by:

$$
\pi_n  = \frac{(\nu+1)\sigma^2}{\nu\sigma^2+ \pare{t^{(n)}-{\xvect}^{(n)}\wvec^{(t)}}^2}
$$

However, the term $(\nu+1)\sigma^2$ can be dropped from the numerator in $\pi$ because it is just a constant scaling the gradient wrt $\wvec$. This constant does not change the minimum, so the resulting $\wvec$ from this algorithm is equivalent to that obtained within the probabilistic interpretation of the Student-t distribution.

This is much faster than using plain gradient descent (or should be) because gradient descent only uses first-order information about the loss function to minimize, while exact coordinate steps exploit the full curvature at each subproblem.

#### Coordinate update for $\sigma^2_{\text{opt}}$

The previous subsection about the coordinate descent steps to solve for $\wvec$ showed that we need an iterative algorithm, since $\pi$ depends on the current $\wvec$ (and $\sigma^2$). This directly connects with the observation made in the previous subsection, where the optimal value for $\wvec$  depends on the current value of $\sigma^2$. This means that we need to update both parameters at the same time to reach the minimum.

It would be nice to obtain a closed-form coordinate optimization step for $\sigma^2$, to avoid a gradient descent step. It turns out we can. The E-step is computed only once per iteration, using the values $\wvec^{(t)},\sigma^{2(t)},\nu^{(t)}$ available at the start of the iteration; that same $\pi_n$ is then reused for both the $\wvec$ update and the $\sigma^2$ update below, it is never recomputed with the freshly updated $\wvec^{(t+1)}$. Given the just-updated $\wvec^{(t+1)}$ and this same $\pi_n$, the coordinate update (M step) for $\sigma^2$ is:

$$
 \sigma^{2(t+1)} = \frac{1}{N}\sum_{n=1}^N \pi_n \pare{t^{(n)}-{\xvect}^{(n)}\wvec^{(t+1)}}^2
$$

This is known as the Expectation Conditional (or coordinate) maximization algorithm. One common E-step is followed by coordinate steps on groups of parameters rather than a single maximization step. Why? Because it is not possible to obtain a closed-form update for all parameters at the same time. Note that the coordinate update for $\sigma^2$ reuses the just-updated $\wvec^{(t+1)}$ inside the residual, but $\pi_n$ itself is the same one computed before the $\wvec$ update.

Here, $\pi_n$ must be the original one from the E-step, evaluated at $\wvec^{(t)},\nu^{(t)},\sigma^{2(t)}$ (i.e. before this iteration's updates), and not the same quantity used for the $\wvec$ update above:

$$
\pi_n = \frac{(\nu^{(t)}+1)\sigma^{2(t)}}{\nu^{(t)}\sigma^{2(t)}+\pare{t^{(n)}-{\xvect}^{(n)}\wvec^{(t)}}^2}
$$

For this formula to actually be the M-step, $\pi_n$ must be the exact E-step weight from the Gaussian scale-mixture representation of the Student-t.

Importantly, in contrast to $\wvec$, there is no way (or at least I do not know how) to obtain the coordinate update for $\sigma^2$ directly from rewriting the gradient as we did with $\wvec$.

The algorithm would then be updated as follows:

1. With $\wvec^{(t)},\sigma^{2(t)},\nu^{(t)}$, compute the E-step weight, once: $\pi_n=\dfrac{(\nu^{(t)}+1)\sigma^{2(t)}}{\nu^{(t)}\sigma^{2(t)}+\pare{t^{(n)}-{\xvect}^{(n)}\wvec^{(t)}}^2}$.
2. Using $\pi_n$, update $\wvec^{(t+1)} = \pareinv{\Xmatt\mathbf{\Pi}\Xmat}\Xmatt\mathbf{\Pi}\tvec$.
3. Update $\sigma^{2(t+1)} = \dfrac{1}{N}\sum_{n=1}^N \pi_n\pare{t^{(n)}-{\xvect}^{(n)}\wvec^{(t+1)}}^2$.
4. Repeat, with $t\leftarrow t+1$, until convergence.

#### Optimizing wrt $\nu$.

It turns out that the fastest option to update this parameter is through what is known as the Expectation/Conditional Maximization Either algorithm. Basically, we perform the E step, run the M step on some parameters, and maximize the exact likelihood wrt others. This is described in section 6 from the reference outlined before.

We have already done this in the section where we obtained the gradient wrt $\nu$:


$$
\frac{\partial}{\partial\nu}\log p(\tvec\mid \Xmat,\wvec,\sigma^2,\nu) = \frac{N}{2}\psi\!\left(\frac{\nu+1}{2}\right) - \frac{N}{2}\psi\!\left(\frac{\nu}{2}\right) - \frac{N}{2\nu} - \frac12\sum_{n=1}^N\log\!\left(1+\frac{\pare{t^{(n)}-{\xvect}^{(n)}\wvec}^2}{\nu\sigma^2}\right) + \frac{\nu+1}{2\nu}\sum_{n=1}^N\frac{\pare{t^{(n)}-{\xvect}^{(n)}\wvec}^2}{\nu\sigma^2+\pare{t^{(n)}-{\xvect}^{(n)}\wvec}^2}
$$

This expression can be shown to be equivalent to (Equation 30 in the paper):

$$
\begin{split}
&\frac{\partial}{\partial\nu}\log p(\tvec\mid \Xmat,\wvec,\sigma^2,\nu) =\psi\pare{\frac{\nu+1}{2}} - \psi\pare{\frac{\nu}{2}} + 1 + \log\frac{\nu}{\nu+1} + \frac1N\sum_{n=1}^N\bra{\log \pi_n - \pi_n}\\
&\pi_n = \dfrac{(\nu+1)\sigma^2}{\nu\sigma^2+\pare{t^{(n)}-{\xvect}^{(n)}\wvec}^2}
\end{split}
$$

This equivalence can be easily obtained by asking Claude to perform the equivalence. There are simple tricks. Why did the authors decide to write this in this form? Well, because this form shows a type of function that can be solved using search methods such as the bisection method (half interval search), as discussed in the paper. The one-dimensional function over $\nu$ results from fixing values in the gradient: since the E-step is computed only once per iteration and both $\wvec^{(t+1)}$ and $\sigma^{2(t+1)}$ are already available from the previous two subsections, we fix them, and we are left with a function of $\nu$ alone:

$$
\begin{split}
&\frac{\partial}{\partial\nu}\log p(\tvec\mid \Xmat,\wvec^{(t+1)},\sigma^{2(t+1)},\nu) =\psi\pare{\frac{\nu+1}{2}} - \psi\pare{\frac{\nu}{2}} + 1 + \log\frac{\nu}{\nu+1} + \frac1N\sum_{n=1}^N\bra{\log \pi_n - \pi_n}\\
&\pi_n = \dfrac{(\nu+1)\sigma^{2(t+1)}}{\nu\sigma^{2(t+1)}+\pare{t^{(n)}-{\xvect}^{(n)}\wvec^{(t+1)}}^2}
\end{split}
$$

Setting this to zero and solving for $\nu$ gives $\nu^{(t+1)}$.

### (Exercise) Optimization through a variant using the ideas introduced so far

It turns out that the best way to fit the Student-t is through the EM algorithm described so far. However, here is an exercise one can do to get a deep understanding of how one should choose different optimization techniques depending on the type of problem we have, and what to think when a new problem arises. Even more in this new era of AI, where coding is delegated to a machine, there is more time to think and to decide which algorithm is the best to use. In real life, the limitation we used to have to implement an algorithm in a reasonable amount of time to be profitable for a company is reduced. So nowadays, given a problem, we are not so restricted in using algorithmic implementations from libraries and can pivot to custom implementations.

I suggest trying the following coordinate update algorithm. Suppose that, for whatever reason, there is no EM algorithm for your idea or you do not know how to derive it. Here is an algorithm that should go faster than naive gradient descent on the three parameters.

* Optimize $\wvec$ using gradient descent or by the reweighting interpretation performed by the IRLS/EM algorithm.
* By noting that the gradient over $\sigma$ is a one-dimensional function, try to see if  this function satisfies the conditions to use a bisection method (such as monotonicity). Solve $\sigma$ given \nu and $\wvec$ fixed using this method.
* For $\nu$ we have already shown it can be solved through a bisection method.

Compare with the naive gradient descent update.

### A caveat: the MLE of $\nu$ can diverge to infinity

The bisection step for $\nu$ (`student_t_m_step_nu`) can raise an error such as *"f(a) and f(b) must have different signs"* on some datasets. This is not a bug: it happens whenever the data does not actually need heavy tails, and it is worth understanding why.

Recall that as $\nu\to\infty$ the Student-t density converges to a Gaussian $\mathcal{N}(\mu,\sigma^2)$. So the Gaussian sits at the boundary ($\nu=\infty$) of the Student-t family. If the residuals are genuinely (close to) Gaussian, with no real outliers, that boundary point is the best-fitting member of the whole family — no finite $\nu$ can do better.

We can see this directly from the score function we already derived and implemented (`student_t_nu_score`, matching Equation 30 of Liu and Rubin):

$$
s(\nu) = \psi\!\left(\frac{\nu+1}{2}\right) - \psi\!\left(\frac{\nu}{2}\right) + 1 + \log\frac{\nu}{\nu+1} + \frac1N\sum_{n=1}^N\left[\log w_n - w_n\right]
$$

with $w_n=\frac{(\nu+1)\sigma^2}{\nu\sigma^2+r_n^2}$. This is (proportional to) $\frac{\partial}{\partial\nu}\log p$, holding $\wvec,\sigma^2$ fixed. If, for a given dataset, $s(\nu)>0$ for every finite $\nu$ (which is exactly what happens when there are no outliers), then the log-likelihood is strictly increasing in $\nu$ over the whole range $(0,\infty)$: increasing $\nu$ always helps, so its supremum is only attained in the limit $\nu\to\infty$, and no finite maximizer exists. This is precisely why the bisection bracket never contains a sign change in that case.

The statistical intuition behind why $s(\nu)$ stays positive: making the tails heavier (decreasing $\nu$) is a model that predicts more large residuals than actually occur. If the residuals are already well explained by a Gaussian (no excess kurtosis), spending model flexibility on heavy tails does not help — it only pulls probability mass away from the bulk of the data towards tails where there is nothing to explain, which costs likelihood rather than gaining it. The likelihood therefore favors the least "heavy-tailed" member of the family it can reach, pushing $\nu\to\infty$.

This is a known and documented degeneracy of the Student-t MLE (not specific to our implementation): when the empirical excess kurtosis of the residuals is low, $\hat\nu$ is unbounded. In practice, this means `student_t_m_step_nu` should either be called with a generous but finite upper bracket (accepting that a very large returned $\nu$ effectively means "the data looks Gaussian, no need for robustness here"), or the search should be abandoned in favor of capping $\nu$ once the data shows no evidence of outliers.

## Robustness to outliers

We shall ask ourselves why we care about deriving loss functions that depend on the Student-t and Laplace distributions. The reason is that these distributions are more robust to "rare" points, usually called outliers. 

The term outlier is a misconception. It usually stands for points that do not follow the tendency of what we want to model. There are some "stats" tricks such as removing points that go beyond the interquartile range. However, I totally disagree with this. An outlier is only an outlier when there is clear evidence, based on our knowledge, that the point is introducing wrong information into what we are modelling. If this is not the case, it is not the data, but the model, what we have to change, for instance by using mixture distributions.

Think, for example, of house price prediction. It might be the case that in a dataset we have a point that says that we have a 200 square meter house with 50 rooms. This is probably something wrong because that would imply each room is 4 square meters, which looks more like a labelling error than a real house. Probably the house had 5 rooms and whoever labelled this house made an error. This error can be as easily made as just clicking 0 by chance when introducing the house information into the house announcement webpage.

On the other hand, we might be modelling the price of a house. Think, for example, in a luxury house dataset, where 200 square meter houses had a price around 2 million euros. It might be the case that we have a couple of 200 square meter houses having a price around 10 million euros. The price is modelled from things such as the number of rooms, the location, the size of the house, etc. Attending to some standard ways of analyzing outliers (such as the interquartile range), we might be removing these points just because they are going to bias our predictions. This is totally unacceptable. In luxury houses it might be the case that a house has a very expensive material or tons of tiger rugs, which are very expensive. This is clearly not an outlier but something we must model. Let's see how we can use these likelihoods to be robust to these situations.

We now analyze how the choice of observation model, Gaussian, Laplace, or Student-t, influences a maximum likelihood fit when the data contains an outlier.

To make the three likelihoods comparable, we fix all of them to have **mean $0$** and **variance $1$**. For the Student-t, whose shape also depends on the degrees of freedom $\nu$, we additionally study $\nu=1,3,30$: this lets us see how, even after matching the first two moments, the value of $\nu$ still changes how the distribution reacts to an outlier.

Below we state the mean and variance of each distribution, say which parameters they depend on, and fix those parameters to match the setup above.

- **Gaussian**: parameters are location $\mu$ and variance $\sigma^2$. Mean $\mathbb{E}[x]=\mu$ depends only on $\mu$; variance $\mathbb{V}[x]=\sigma^2$ depends only on $\sigma^2$. We set $\mu=0,\ \sigma^2=1$.
- **Laplace**: parameters are location $\mu$ and scale $b>0$. Mean $\mathbb{E}[x]=\mu$ depends only on $\mu$; variance $\mathbb{V}[x]=2b^2$ depends only on $b$. We set $\mu=0$ and $b=1/\sqrt{2}$, so that $\mathbb{V}[x]=2b^2=1$.
- **Student-t**: parameters are location $\mu$, scale $\sigma>0$ and degrees of freedom $\nu>0$. Mean $\mathbb{E}[x]=\mu$ for $\nu>1$ depends only on $\mu$ (undefined for $\nu\leq 1$); variance $\mathbb{V}[x]=\frac{\nu}{\nu-2}\sigma^2$ for $\nu>2$ depends on $\sigma$ and $\nu$ (infinite for $1<\nu\leq 2$, undefined for $\nu\leq 1$). We set $\mu=0$ for all cases. For $\nu=3,30$ we solve $\sigma^2=\frac{\nu-2}{\nu}$ so that $\mathbb{V}[x]=1$, which gives $\sigma\approx0.577$ for $\nu=3$ and $\sigma\approx0.966$ for $\nu=30$. For $\nu=1$ (the Cauchy distribution), the variance is not finite for any $\sigma$, so we cannot match it to $1$; we simply set $\sigma=1$, keeping in mind that this case has, by construction, no finite variance.

### Visualizing the three densities

We now plot the three densities using the parameters fixed above: Gaussian ($\sigma=1$), Laplace ($b=1/\sqrt{2}$) and Student-t for $\nu=1,3,30$ (with $\sigma$ matched to unit variance whenever it exists).


In [ ]:
%matplotlib inline
plt.close("all")

x_grid = np.linspace(-8, 8, 1000)

mu = 0.0
sigma_gauss = 1.0
b_laplace = 1.0 / np.sqrt(2.0)
nu_values = [1, 3, 30,10000]
sigma_t = {nu: (np.sqrt((nu - 2) / nu) if nu > 2 else 1.0) for nu in nu_values}

pdf_gauss = scipy_norm.pdf(x_grid, loc=mu, scale=sigma_gauss)
pdf_laplace = scipy_laplace.pdf(x_grid, loc=mu, scale=b_laplace)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax in axes:
    ax.plot(x_grid, pdf_gauss, color='C0', linewidth=3.5, label='Gaussian',linestyle='-.')
    ax.plot(x_grid, pdf_laplace, color='C1', linewidth=3.5, linestyle='--', label='Laplace')
    color_counter = 2
    for nu in nu_values:
        pdf_t = scipy_t.pdf(x_grid, df=nu, loc=mu, scale=sigma_t[nu])
        ax.plot(x_grid, pdf_t, linewidth=2, label=fr'Student-t ($\nu={nu}$)', color=f'C{color_counter}')
        color_counter+=1
    ax.set_xlabel('$x$')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('$p(x)$')
axes[0].set_title('Linear scale')
axes[1].set_yscale('log')
axes[1].set_ylim(1e-6, 1)
axes[1].set_title('Log scale (tails)')
axes[1].legend(loc='upper right', fontsize=9)

fig.suptitle('Gaussian vs Laplace vs Student-t densities (mean 0, variance 1 where defined)')
plt.tight_layout()
plt.show()


On the log scale, it becomes clear that the Gaussian tail decays much faster than the Laplace tail, which in turn decays much faster than any Student-t tail. Among the Student-t curves, smaller $\nu$ means heavier tails (more probability mass far from $\mu$), even though the variance has been matched to $1$ whenever possible. In the limit of $\nu$ being infinite, we have a Gaussian.

### Dataset with an outlier

We will look at the influence of fitting these distributions via maximum likelihood. To do so, we generate data from a Gaussian distribution with mean $0$ and variance $1$, and we generate one far-away point (an outlier).

In [ ]:
%matplotlib inline
plt.close("all")

rng = np.random.default_rng(0)

mu_true, sigma_true = 0.0, 1.0
N = 15

x_clean = rng.normal(mu_true, sigma_true, size=N)

outlier_value = 10.0
x_with_outlier = np.append(x_clean, outlier_value)

fig, ax = plt.subplots(1, 1, figsize=(9, 2.5))

y_jitter = np.zeros_like(x_clean)
ax.scatter(x_clean, y_jitter, color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter([outlier_value], [0], color='tab:red', marker='X', s=140, zorder=3, label='outlier')

ax.axvline(mu_true, color='k', linestyle=':', linewidth=1, label=fr'true $\mu={mu_true}$')

ax.set_yticks([])
ax.set_xlabel('$x$')
ax.set_title('1D dataset: inliers plus one outlier')
ax.legend(loc='upper left')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


We now plot each of the distributions with the parameters mentioned and calculate the log-likelihood

In [ ]:
%matplotlib inline
plt.close("all")

x_grid = np.linspace(-6, 12, 1000)

loglik_gauss = scipy_norm.logpdf(x_with_outlier, loc=mu, scale=sigma_gauss).sum()
loglik_laplace = scipy_laplace.logpdf(x_with_outlier, loc=mu, scale=b_laplace).sum()
loglik_t = {nu: scipy_t.logpdf(x_with_outlier, df=nu, loc=mu, scale=sigma_t[nu]).sum() for nu in nu_values}

fig, ax = plt.subplots(1, 1, figsize=(11, 5))

ax.plot(x_grid, scipy_norm.pdf(x_grid, loc=mu, scale=sigma_gauss), color='C0', linewidth=3.5, linestyle='-.',
        label=f'Gaussian (log-lik={loglik_gauss:.1f})')
ax.plot(x_grid, scipy_laplace.pdf(x_grid, loc=mu, scale=b_laplace), color='C1', linewidth=3.5, linestyle='--',
        label=f'Laplace (log-lik={loglik_laplace:.1f})')

color_counter = 2
for nu in nu_values:
    ax.plot(x_grid, scipy_t.pdf(x_grid, df=nu, loc=mu, scale=sigma_t[nu]), linewidth=2, color=f'C{color_counter}',
            label=fr'Student-t ($\nu={nu}$, log-lik={loglik_t[nu]:.1f})')
    color_counter += 1

ax.scatter(x_clean, np.zeros_like(x_clean), color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter([outlier_value], [0], color='tab:red', marker='X', s=140, zorder=3, label='outlier')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Densities evaluated on the dataset with an outlier')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


As expected, the Gaussian gets by far the worst (most negative) total log-likelihood, since its light tails penalize the outlier very heavily, while the Cauchy ($\nu=1$) gets the best one, as its heavy tails make the outlier far less surprising; Laplace and the milder Student-t settings ($\nu=3,30$) fall in between.

### Maximum likelihood fit: Gaussian vs Laplace

We now fit $\mu$ (and the scale) of the Gaussian and the Laplace to the dataset with the outlier via maximum likelihood, and plot the resulting distributions on top of the data. For the Gaussian, the MLE is $\hat\mu=\bar{x}$ and $\hat\sigma^2=\frac{1}{N}\sum_n(x^n-\hat\mu)^2$. For the Laplace, the MLE is $\hat\mu=\text{median}(x)$ and $\hat{b}=\frac{1}{N}\sum_n|x^n-\hat\mu|$.

In [ ]:
%matplotlib inline
plt.close("all")

x_grid = np.linspace(-6, 12, 1000)

mu_mle_gauss = x_with_outlier.mean()
sigma2_mle_gauss = np.mean((x_with_outlier - mu_mle_gauss)**2)

mu_mle_laplace = np.median(x_with_outlier)
b_mle_laplace = np.mean(np.abs(x_with_outlier - mu_mle_laplace))

fig, ax = plt.subplots(1, 1, figsize=(11, 5))

ax.plot(x_grid, scipy_norm.pdf(x_grid, loc=mu_mle_gauss, scale=np.sqrt(sigma2_mle_gauss)), color='C0', linewidth=3.5,
        linestyle='-.', label=fr'Gaussian MLE ($\hat\mu={mu_mle_gauss:.2f}$, $\hat\sigma^2={sigma2_mle_gauss:.2f}$)')
ax.plot(x_grid, scipy_laplace.pdf(x_grid, loc=mu_mle_laplace, scale=b_mle_laplace), color='C1', linewidth=3.5,
        linestyle='--', label=fr'Laplace MLE ($\hat\mu={mu_mle_laplace:.2f}$, $\hat{{b}}={b_mle_laplace:.2f}$)')

ax.scatter(x_clean, np.zeros_like(x_clean), color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter([outlier_value], [0], color='tab:red', marker='X', s=140, zorder=3, label='outlier')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian vs Laplace: maximum likelihood fit on the dataset with an outlier')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


Let's now add more outliers

In [ ]:
%matplotlib inline
plt.close("all")

x_grid = np.linspace(-6, 14, 1000)

outlier_values = np.array([10.0, 12.0, 9.0])
x_multi_outlier = np.append(x_clean, outlier_values)

mu_mle_gauss_multi = x_multi_outlier.mean()
sigma2_mle_gauss_multi = np.mean((x_multi_outlier - mu_mle_gauss_multi)**2)

mu_mle_laplace_multi = np.median(x_multi_outlier)
b_mle_laplace_multi = np.mean(np.abs(x_multi_outlier - mu_mle_laplace_multi))

fig, ax = plt.subplots(1, 1, figsize=(11, 5))

ax.plot(x_grid, scipy_norm.pdf(x_grid, loc=mu_mle_gauss_multi, scale=np.sqrt(sigma2_mle_gauss_multi)), color='C0',
        linewidth=3.5, linestyle='-.',
        label=fr'Gaussian MLE ($\hat\mu={mu_mle_gauss_multi:.2f}$, $\hat\sigma^2={sigma2_mle_gauss_multi:.2f}$)')
ax.plot(x_grid, scipy_laplace.pdf(x_grid, loc=mu_mle_laplace_multi, scale=b_mle_laplace_multi), color='C1',
        linewidth=3.5, linestyle='--',
        label=fr'Laplace MLE ($\hat\mu={mu_mle_laplace_multi:.2f}$, $\hat{{b}}={b_mle_laplace_multi:.2f}$)')

ax.scatter(x_clean, np.zeros_like(x_clean), color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter(outlier_values, np.zeros_like(outlier_values), color='tab:red', marker='X', s=140, zorder=3, label='outliers')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian vs Laplace: maximum likelihood fit with three outliers')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


We observe how the Gaussian is moved towards explaining the outliers. Since the tails are flat, it does so by increasing the variance and moving the mean, moving most of the probability mass out of where most of the distribution is present.

### Maximum likelihood Student-t fixing $\nu$

The algorithm used to do maximum likelihood for the Student-t given a fixed $\nu$ is the EM algorithm (Liu and Rubin, 1995, *ML estimation of the t distribution using EM and its extensions*, Section 4). In reality is a coordinate EM algorithm, since the M-step is performed on each coordinate. Its update equations, particularized to our 1D fully-observed case, are:

E-step (computed once per iteration, with $\mu^{(t)},\sigma^{2(t)}$):

$$
\begin{split}
\pi_n = \frac{(\nu+1)\sigma^{2(t)}}{\nu\sigma^{2(t)}+\pare{x^{(n)}-\mu^{(t)}}^2}
\end{split}
$$

Coordinate M-step (reusing the same $\pi_n$ for both updates):

$$
\begin{split}
\mu^{(t+1)} &= \frac{\sum^N_{n=1} \pi_n x^{(n)}}{\sum^N_{n=1} \pi_n}\\
\sigma^{2(t+1)} &= \frac{1}{N}\sum^N_{n=1} \pi_n\pare{x^{(n)}-\mu^{(t+1)}}^2
\end{split}
$$

Note this equations are pretty similar to the ones studied before, with the difference that the mean is a parameter itself rather than being computed through a linear operator $\xvect\wvec$.

In [ ]:
def student_t_e_step_loc_scale(x, mu, sigma2, nu):
    return (nu + 1) / (nu + (x - mu)**2 / sigma2)

def student_t_m_step_loc_scale(x, w):
    mu = np.sum(w * x) / np.sum(w)
    sigma2 = np.mean(w * (x - mu)**2)
    return mu, sigma2

def fit_student_t(x, nu, n_iter=2000):
    mu, sigma2 = np.median(x), np.var(x)
    for _ in range(n_iter):
        w = student_t_e_step_loc_scale(x, mu, sigma2, nu)
        mu, sigma2 = student_t_m_step_loc_scale(x, w)
    return mu, sigma2

mle_t = {nu: fit_student_t(x_with_outlier, nu) for nu in nu_values}

In [ ]:
%matplotlib inline
plt.close("all")

fig, ax = plt.subplots(1, 1, figsize=(11, 5))

ax.plot(x_grid, scipy_norm.pdf(x_grid, loc=mu_mle_gauss, scale=np.sqrt(sigma2_mle_gauss)), color='C0', linewidth=3.5,
        linestyle='-.', label=fr'Gaussian MLE ($\hat\mu={mu_mle_gauss:.2f}$, $\hat\sigma^2={sigma2_mle_gauss:.2f}$)')
ax.plot(x_grid, scipy_laplace.pdf(x_grid, loc=mu_mle_laplace, scale=b_mle_laplace), color='C1', linewidth=3.5,
        linestyle='--', label=fr'Laplace MLE ($\hat\mu={mu_mle_laplace:.2f}$, $\hat{{b}}={b_mle_laplace:.2f}$)')

color_counter = 2
for nu in nu_values:
    mu_mle_t, sigma2_mle_t = mle_t[nu]
    ax.plot(x_grid, scipy_t.pdf(x_grid, df=nu, loc=mu_mle_t, scale=np.sqrt(sigma2_mle_t)), linewidth=2, color=f'C{color_counter}',
            label=fr'Student-t MLE ($\nu={nu}$, $\hat\mu={mu_mle_t:.2f}$, $\hat\sigma^2={sigma2_mle_t:.2f}$)')
    color_counter += 1

ax.scatter(x_clean, np.zeros_like(x_clean), color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter([outlier_value], [0], color='tab:red', marker='X', s=140, zorder=3, label='outlier')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian, Laplace and Student-t: maximum likelihood fit on the dataset with an outlier')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


With more outliers.

In [ ]:
%matplotlib inline
plt.close("all")

mle_t_multi = {nu: fit_student_t(x_multi_outlier, nu) for nu in nu_values}

fig, ax = plt.subplots(1, 1, figsize=(11, 5))

ax.plot(x_grid, scipy_norm.pdf(x_grid, loc=mu_mle_gauss_multi, scale=np.sqrt(sigma2_mle_gauss_multi)), color='C0',
        linewidth=3.5, linestyle='-.',
        label=fr'Gaussian MLE ($\hat\mu={mu_mle_gauss_multi:.2f}$, $\hat\sigma^2={sigma2_mle_gauss_multi:.2f}$)')
ax.plot(x_grid, scipy_laplace.pdf(x_grid, loc=mu_mle_laplace_multi, scale=b_mle_laplace_multi), color='C1',
        linewidth=3.5, linestyle='--',
        label=fr'Laplace MLE ($\hat\mu={mu_mle_laplace_multi:.2f}$, $\hat{{b}}={b_mle_laplace_multi:.2f}$)')

color_counter = 2
for nu in nu_values:
    mu_mle_t, sigma2_mle_t = mle_t_multi[nu]
    ax.plot(x_grid, scipy_t.pdf(x_grid, df=nu, loc=mu_mle_t, scale=np.sqrt(sigma2_mle_t)), linewidth=2, color=f'C{color_counter}',
            label=fr'Student-t MLE ($\nu={nu}$, $\hat\mu={mu_mle_t:.2f}$, $\hat\sigma^2={sigma2_mle_t:.2f}$)')
    color_counter += 1

ax.scatter(x_clean, np.zeros_like(x_clean), color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter(outlier_values, np.zeros_like(outlier_values), color='tab:red', marker='X', s=140, zorder=3, label='outliers')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian, Laplace and Student-t: maximum likelihood fit with three outliers')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


We observe how the bigger the degrees of freedom $\nu$ the more Gaussian the student-t becomes, and the less robust is towards contaminating data (see for instance purple and brown curves).

### Maximum likelihood Student-t learning all parameters

To also learn $\nu$, instead of fixing it, we use the ECME algorithm (Liu and Rubin, 1995, Section 6). Again, this is pretty similar to what we have been discussing so far. This algorithm uses an E step followed by coordinate M steps on the mean and the scale parameter. The $\nu$ parameter is optimized by coordinate ascent directly on the log-likelihood function. In summary, we have:

E-step:

$$
\begin{split}
\pi_n = \frac{(\nu^{(t)}+1)\sigma^{2(t)}}{\nu^{(t)}\sigma^{2(t)}+\pare{x^{(n)}-\mu^{(t)}}^2}
\end{split}
$$

CM-step 1:

$$
\begin{split}
\mu^{(t+1)} &= \frac{\sum^N_{n=1} \pi_n x^{(n)}}{\sum^N_{n=1} \pi_n}\\
\sigma^{2(t+1)} &= \frac{1}{N}\sum^N_{n=1} \pi_n\pare{x^{(n)}-\mu^{(t+1)}}^2
\end{split}
$$

CM-step 2 is a coordinate update on the actual log-likelihood: fixing $\mu=\mu^{(t+1)},\sigma^2=\sigma^{2(t+1)}$ at their CM-step 1 values, differentiating the log-likelihood with respect to $\nu$ and setting it to zero gives Rubin's equation, which is solved numerically for $\nu^{(t+1)}$:

$$
\begin{split}
-\psi\pare{\frac{\nu}{2}}+\ln\pare{\frac{\nu}{2}}+1+\frac{1}{N}\sum^N_{n=1}\bra{\ln \pi_n(\nu) - \pi_n(\nu)}+\psi\pare{\frac{\nu+1}{2}}-\ln\pare{\frac{\nu+1}{2}} = 0,\qquad \pi_n(\nu) = \frac{(\nu+1)\sigma^{2(t+1)}}{\nu\sigma^{2(t+1)}+\pare{x^{(n)}-\mu^{(t+1)}}^2}
\end{split}
$$

where $\psi$ is the digamma function. Rubin solves this with a one-dimensional search, specifically the half-interval method (Carnahan, Luther and Wilkes, 1969), i.e., bisection. In `scipy`, $\psi$ is `scipy.special.digamma`, and the bisection search is `scipy.optimize.brentq` (or `bisect`), given a bracket $[\nu_{\min},\nu_{\max}]$ in which the left-hand side changes sign. This is similar to what we have explained before.

In [ ]:
def nu_score(nu, x, mu, sigma2):
    w = student_t_e_step_loc_scale(x, mu, sigma2, nu)
    return (-digamma(nu / 2) + np.log(nu / 2) + 1
            + np.mean(np.log(w) - w)
            + digamma((nu + 1) / 2) - np.log((nu + 1) / 2))

def student_t_cm_step_nu(x, mu, sigma2, bracket=(1e-3, 1000.0)):
    return brentq(nu_score, bracket[0], bracket[1], args=(x, mu, sigma2))

def fit_student_t_ecme(x, nu_init=5.0, n_iter=100, nu_bracket=(1e-3, 1000.0)):
    mu, sigma2, nu = np.median(x), np.var(x), nu_init
    for _ in range(n_iter):
        w = student_t_e_step_loc_scale(x, mu, sigma2, nu)
        mu, sigma2 = student_t_m_step_loc_scale(x, w)
        nu = student_t_cm_step_nu(x, mu, sigma2, nu_bracket)
    return mu, sigma2, nu

In [ ]:
mle_ecme_single = fit_student_t_ecme(x_with_outlier)
mle_ecme_multi = fit_student_t_ecme(x_multi_outlier)

print(f"One outlier:    mu={mle_ecme_single[0]:.2f}, sigma2={mle_ecme_single[1]:.2f}, nu={mle_ecme_single[2]:.2f}")
print(f"Three outliers: mu={mle_ecme_multi[0]:.2f}, sigma2={mle_ecme_multi[1]:.2f}, nu={mle_ecme_multi[2]:.2f}")


In [ ]:
%matplotlib inline
plt.close("all")

mu_ecme, sigma2_ecme, nu_ecme = mle_ecme_single

fig, ax = plt.subplots(1, 1, figsize=(11, 5))

ax.plot(x_grid, scipy_norm.pdf(x_grid, loc=mu_mle_gauss, scale=np.sqrt(sigma2_mle_gauss)), color='C0', linewidth=3.5,
        linestyle='-.', label=fr'Gaussian MLE ($\hat\mu={mu_mle_gauss:.2f}$, $\hat\sigma^2={sigma2_mle_gauss:.2f}$)')
ax.plot(x_grid, scipy_t.pdf(x_grid, df=nu_ecme, loc=mu_ecme, scale=np.sqrt(sigma2_ecme)), color='C2', linewidth=2,
        label=fr'Student-t ECME ($\hat\mu={mu_ecme:.2f}$, $\hat\sigma^2={sigma2_ecme:.2f}$, $\hat\nu={nu_ecme:.2f}$)')

ax.scatter(x_clean, np.zeros_like(x_clean), color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter([outlier_value], [0], color='tab:red', marker='X', s=140, zorder=3, label='outlier')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian vs Student-t (ECME, learning all): maximum likelihood fit with one outlier')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
%matplotlib inline
plt.close("all")

mu_ecme_multi, sigma2_ecme_multi, nu_ecme_multi = mle_ecme_multi

fig, ax = plt.subplots(1, 1, figsize=(11, 5))

ax.plot(x_grid, scipy_norm.pdf(x_grid, loc=mu_mle_gauss_multi, scale=np.sqrt(sigma2_mle_gauss_multi)), color='C0',
        linewidth=3.5, linestyle='-.',
        label=fr'Gaussian MLE ($\hat\mu={mu_mle_gauss_multi:.2f}$, $\hat\sigma^2={sigma2_mle_gauss_multi:.2f}$)')
ax.plot(x_grid, scipy_t.pdf(x_grid, df=nu_ecme_multi, loc=mu_ecme_multi, scale=np.sqrt(sigma2_ecme_multi)), color='C2',
        linewidth=2,
        label=fr'Student-t ECME ($\hat\mu={mu_ecme_multi:.2f}$, $\hat\sigma^2={sigma2_ecme_multi:.2f}$, $\hat\nu={nu_ecme_multi:.2f}$)')

ax.scatter(x_clean, np.zeros_like(x_clean), color='tab:blue', s=60, zorder=3, label='inliers')
ax.scatter(outlier_values, np.zeros_like(outlier_values), color='tab:red', marker='X', s=140, zorder=3, label='outliers')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian vs Student-t (ECME, learning all): maximum likelihood fit with three outliers')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


Importantly,  when learning $\nu$, it goes from $\hat\nu\approx1.42$ with one outlier to $\hat\nu\approx0.87$ with three outliers, i.e., the more contamination in the data, the heavier the tails the algorithm chooses for itself. This is important. While we could choose $\nu$ manually if we know the data is contaminated, to be robust to outliers, maximizing the actual likelihood over $\nu$ already discovers how heavy the tails need to be to accommodate the outliers.

## Robustifying Linear Regression

Let's see how all the machinery derived so far can be used within linear regression models. First of all, let's simulate the house price scenario with tiger rugs.

In [ ]:
np.random.seed(42)

# Normal data
x = np.linspace(0, 10, 40)
noise = np.random.normal(0, 0.8, size=x.shape)
y = 2*x + 1 + noise

# Outliers
x_out = np.array([8.0, 9.0, 10.0])
y_out = np.array([45, 55, 65])

# Final dataset
X = np.concatenate([x, x_out])
Y = np.concatenate([y, y_out])

plt.figure(figsize=(7,5))
plt.scatter(X, Y)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dataset with outliers")
plt.grid(True)
plt.show()

### Robustness comparison: SSE cost vs. absolute cost vs. Student-t cost (various $\nu$)
Let's first visualize the loss function as a function of the weights. We fix $b=0$ and $\sigma^2=1$, and vary $w$ over the dataset with outliers already generated. We compare the SSE cost, $\sum_n (t^{(n)} - wx^{(n)})^2$, the absolute cost, $\sum_n |t^{(n)} - wx^{(n)}|$, and the Student-t cost, $\sum_n \log\left(1+\frac{(t^{(n)}-wx^{(n)})^2}{\nu\sigma^2}\right)$, for $\nu \in \{1,2,8,10000\}$. Since all the curves live on different scales, we normalize each one by dividing by its own minimum, so that they all start at 1 and are directly comparable on the same axis: we see how the SSE grows much faster (more sensitive to outliers), the absolute cost grows linearly (much smoother), and the Student-t cost progressively approaches the SSE's behavior as $\nu$ grows. 

The fact that the squared loss increases quadratically vs the Laplace, which is linear, or Student-t, which tends to flatten, implies that for a given weight, the cost when having an outlier increases much more for the SSE. This means that the loss penalizes errors on this data a lot, and thus the weight is moved out of this value to reduce this error.

As we see, the $\wvec$ where each loss achieves the minimum differs. This means that the optimal lines learnt for each dataset will also differ.

In [ ]:
sigma2 = 1
b_fixed = 0.0
nu_values = [1, 2, 8, 10000]

w_range = np.linspace(-5, 10, 400)

sse_cost = np.array([np.sum(squared_loss_function(Y, w * X + b_fixed)) for w in w_range])
sse_norm = sse_cost / sse_cost.min()

abs_cost = np.array([np.sum(absolute_loss_function(Y, w * X + b_fixed)) for w in w_range])
abs_norm = abs_cost / abs_cost.min()

plt.figure(figsize=(7, 5))
plt.plot(w_range, sse_norm, color="black", linewidth=2, label="SSE")
plt.plot(w_range, abs_norm, color="C3", linewidth=2, label="Absolute loss")

for nu, color in zip(nu_values, ["C0", "C1", "C2", "C4", "C5", "C6"]):
    cost = np.array([
        np.sum(student_t_loss_function(Y, w * X + b_fixed, nu, sigma2))
        for w in w_range
    ])
    plt.plot(w_range, cost / cost.min(), linestyle=":", color=color, linewidth=2, label=f"Student-t, $\\nu={nu}$")

plt.xlabel("w")
plt.ylabel("Loss / minimum loss")
plt.title("Relative loss growth as a function of $w$ (b=0, $\\sigma^2=1$)")
plt.legend()
plt.grid(True)
plt.show()

### Fitting via SSE (Gaussian likelihood)

We first fit the optimal line by minimizing the squared error (SSE), which, as we saw in the [regression theory](1_Regression_Shallow_theory.ipynb), has a closed-form solution via ordinary least squares (OLS). Let's first compare what happens when we fit the Gaussian likelihood with and without outliers. We see that outliers clearly bias our line. Here, we could try and fit a mixture of linear models through EM algorithm.

In [ ]:
# design matrix [x, 1] and targets as column vector (full data, with outliers)
X_design = np.stack([X, np.ones_like(X)], axis=1)
T = Y.reshape(-1, 1)

# optimal parameters via OLS (closed-form SSE minimizer)
w_sse, b_sse = fit_norm2_least_square(X_design, T).flatten()

# same fit but excluding the outliers, for comparison
X_design_clean = np.stack([x, np.ones_like(x)], axis=1)
T_clean = y.reshape(-1, 1)
w_sse_clean, b_sse_clean = fit_norm2_least_square(X_design_clean, T_clean).flatten()


print(f"SSE fit (with outliers):  w = {w_sse:.4f}, b = {b_sse:.4f}")
print(f"SSE fit (without outliers):  w = {w_sse_clean:.4f}, b = {b_sse_clean:.4f}")

x_line = np.linspace(X.min(), X.max(), 100)
y_line_sse = w_sse * x_line + b_sse
y_line_sse_clean = w_sse_clean * x_line + b_sse_clean

plt.figure(figsize=(7, 5))
plt.scatter(X, Y, marker="x", label="data")
plt.plot(x_line, y_line_sse, color="C1", linewidth=3, label=f"SSE fit (with outliers): y = {w_sse:.2f}x + {b_sse:.2f}")
plt.plot(x_line, y_line_sse_clean, color="black", linewidth=3, label=f"SSE fit (without outliers): y = {w_sse_clean:.2f}x + {b_sse_clean:.2f}")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Optimal line: SSE (with/without outliers)")
plt.legend()
plt.grid(True)
plt.show()

### Fitting via absolute loss (Laplace likelihood)

As we saw in the [regression theory](1_Regression_Shallow_theory.ipynb), the absolute loss has no closed form, so we fit it via gradient descent. Let's run gradient descent on this model.

In [ ]:
# data in the (N,D) / (D,1) shapes expected by the utils functions
x_data_gd = X.reshape(-1, 1)
t_data_gd = Y.reshape(-1, 1)

# initialize parameters
np.random.seed(0)
w_abs, b_abs = create_computation_graph_linear(1, 1)

# gradient descent hyperparameters
lr = 0.001
epochs = 2000
loss_history = []

for e in range(epochs):

    ## forward pass
    y_pred = computation_graph_linear(x_data_gd, w_abs, b_abs)
    loss = absolute_loss_function(t_data_gd, y_pred)
    loss_history.append(np.sum(loss))

    ## backward pass and update
    grad_w, grad_b = grad_absolute_loss_wrt_linear_model(x_data_gd, t_data_gd, w_abs, b_abs)
    w_abs = w_abs - lr * grad_w
    b_abs = b_abs - lr * grad_b

w_abs, b_abs = w_abs.item(), b_abs.item()

# convergence check
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Absolute loss (sum)")
plt.title("Gradient descent convergence (absolute loss)")
plt.grid(True)
plt.show()

Let's see the line fitted through this loss.

In [ ]:
print(f"SSE fit (with outliers):  w = {w_sse:.4f}, b = {b_sse:.4f}")
print(f"SSE fit (without outliers):  w = {w_sse_clean:.4f}, b = {b_sse_clean:.4f}")
print(f"Absolute loss fit (with outliers, via GD):  w = {w_abs:.4f}, b = {b_abs:.4f}")

x_line = np.linspace(X.min(), X.max(), 100)
y_line_sse = w_sse * x_line + b_sse
y_line_sse_clean = w_sse_clean * x_line + b_sse_clean
y_line_abs = w_abs * x_line + b_abs

plt.figure(figsize=(7, 5))
plt.scatter(X, Y, marker="x", label="data")
plt.plot(x_line, y_line_sse, color="C1", linewidth=3, label=f"SSE fit (with outliers): y = {w_sse:.2f}x + {b_sse:.2f}")
plt.plot(x_line, y_line_sse_clean, color="black", linewidth=3, label=f"SSE fit (without outliers): y = {w_sse_clean:.2f}x + {b_sse_clean:.2f}")
plt.plot(x_line, y_line_abs, color="C2", linewidth=3, label=f"Absolute loss fit: y = {w_abs:.2f}x + {b_abs:.2f}")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Optimal line: SSE (with/without outliers) vs. Absolute loss vs. Student-t")
plt.legend()
plt.grid(True)
plt.show()

We see that the absolute loss is much less sensitive to this outliers. So there is no need to remove the data. If we want a quick model that is not bias by this points, fit a laplace likelihood. Then, you could try and see if it worth using mixture model.

### Fitting via Student-t Likelihood

For the Student-t likelihood, we will analyze different settings. We have seen that the student-t automatically learns $\nu$ on demand, depending on the number of outliers. On the other hand, in contrast to the Laplace and Gaussian likelihood, in the student-t the noise variance plays a role, because optimal $\wvec$ depends on its value. This is not like in the Gaussian or Laplace where we first find optimal $\wvec$ and given this $\wvec$ we learn the aleatoric noise parameter. 

Thus, we first fit the student-t likelihood for several values of $\nu$ through the coordinate EM algorithm desribed, keeping $\nu$ fixed but updating two parameters. **Exercise** compare this case to only fitting $\wvec$ by keeping $\sigma^2$ fixed as well.


In [ ]:
nu_values_fixed = [1, 3, 30]
n_iter_coord = 50

student_t_fits_fixed_nu = {}
student_t_loss_history = {}

for nu_fixed in nu_values_fixed:
    # initialize at the OLS solution, sigma2 at the residual variance
    w_t, b_t = create_computation_graph_linear_from_ols(x_data_gd, t_data_gd)
    r0 = t_data_gd - computation_graph_linear(x_data_gd, w_t, b_t)
    sigma2_t = np.var(r0)

    loss_history = []
    for _ in range(n_iter_coord):
        y_pred = computation_graph_linear(x_data_gd, w_t, b_t)
        loss_history.append(np.sum(student_t_full_loss_function(t_data_gd, y_pred, nu_fixed, sigma2_t)))

        # E-step: computed once per iteration, with the current w,b,sigma2 (before any update)
        pi = student_t_e_step(x_data_gd, t_data_gd, w_t, b_t, nu_fixed, sigma2_t)

        # coordinate update for w,b (weighted least squares), using this pi
        w_t, b_t = student_t_m_step_w(x_data_gd, t_data_gd, pi)

        # coordinate update for sigma2, reusing the same pi (not recomputed)
        sigma2_t = student_t_m_step_sigma2(x_data_gd, t_data_gd, w_t, b_t, pi)

    student_t_fits_fixed_nu[nu_fixed] = (w_t.item(), b_t.item(), sigma2_t.item())
    student_t_loss_history[nu_fixed] = loss_history
    print(f"Student-t (nu={nu_fixed}): w={w_t.item():.4f}, b={b_t.item():.4f}, sigma2={sigma2_t.item():.4f}")



Let's check if the models have converged properly by plotting the loss against iterations. They do. The good point of this coordinate descent/ascent algorithm which do not use partial but full steps is that convergence is always guaranteed. It can be quick or slow, but is always guaranteed.

In [ ]:
plt.figure(figsize=(7, 5))
for nu_fixed in nu_values_fixed:
    plt.plot(student_t_loss_history[nu_fixed], label=fr"$\nu={nu_fixed}$")
plt.xlabel("Iteration")
plt.ylabel("Student-t negative log-likelihood (sum)")
plt.title("Coordinate descent convergence (Student-t, fixed $\\nu$)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
x_line = np.linspace(X.min(), X.max(), 100)

plt.figure(figsize=(7, 5))
plt.scatter(X, Y, marker="x", label="data")
plt.plot(x_line, w_sse * x_line + b_sse, color="C1", linewidth=3, label=f"SSE fit (with outliers): y = {w_sse:.2f}x + {b_sse:.2f}")
plt.plot(x_line, w_abs * x_line + b_abs, color="C2", linewidth=3, label=f"Absolute loss fit: y = {w_abs:.2f}x + {b_abs:.2f}")
for i, nu_fixed in enumerate(nu_values_fixed):
    w_t, b_t, sigma2_t = student_t_fits_fixed_nu[nu_fixed]
    plt.plot(x_line, w_t * x_line + b_t, linewidth=3, color=f"C{3+i}",
             label=fr"Student-t fit ($\nu={nu_fixed}$): y = {w_t:.2f}x + {b_t:.2f}")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Optimal line: SSE vs. Absolute loss vs. Student-t (fixed $\\nu$)")
plt.legend()
plt.grid(True)
plt.show()

We observe how the student with high degrees of freedom startgs being less robust to outliers. Let's see what happens if we fit $\nu$ as well.

In [ ]:
n_iter_ecme = 50

# initialize at the OLS solution, sigma2 at the residual variance, nu at a moderate default
w_ecme, b_ecme = create_computation_graph_linear_from_ols(x_data_gd, t_data_gd)
r0 = t_data_gd - computation_graph_linear(x_data_gd, w_ecme, b_ecme)
sigma2_ecme = np.var(r0)
nu_ecme = 5.0

student_t_ecme_loss_history = []

for _ in range(n_iter_ecme):
    y_pred = computation_graph_linear(x_data_gd, w_ecme, b_ecme)
    student_t_ecme_loss_history.append(np.sum(student_t_full_loss_function(t_data_gd, y_pred, nu_ecme, sigma2_ecme)))

    # E-step: computed once per iteration, with the current w,b,sigma2 (before any update)
    pi = student_t_e_step(x_data_gd, t_data_gd, w_ecme, b_ecme, nu_ecme, sigma2_ecme)

    # coordinate update for w,b (weighted least squares), using this pi
    w_ecme, b_ecme = student_t_m_step_w(x_data_gd, t_data_gd, pi)

    # coordinate update for sigma2, reusing the same pi (not recomputed)
    sigma2_ecme = student_t_m_step_sigma2(x_data_gd, t_data_gd, w_ecme, b_ecme, pi)

    # coordinate update for nu, over the actual likelihood, given the just-updated w,b,sigma2
    nu_ecme = student_t_m_step_nu(x_data_gd, t_data_gd, w_ecme, b_ecme, sigma2_ecme)

print(f"Student-t (learning nu): w={w_ecme.item():.4f}, b={b_ecme.item():.4f}, sigma2={sigma2_ecme.item():.4f}, nu={nu_ecme:.4f}")


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(student_t_ecme_loss_history)
plt.xlabel("Iteration")
plt.ylabel("Student-t negative log-likelihood (sum)")
plt.title("ECME convergence (learning $\\nu$ too)")
plt.grid(True)
plt.show()

In [ ]:
x_line = np.linspace(X.min(), X.max(), 100)

plt.figure(figsize=(7, 5))
plt.scatter(X, Y, marker="x", label="data")
plt.plot(x_line, w_sse * x_line + b_sse, color="C1", linewidth=3, label=f"SSE fit (with outliers): y = {w_sse:.2f}x + {b_sse:.2f}")
plt.plot(x_line, w_abs * x_line + b_abs, color="C2", linewidth=3, label=f"Absolute loss fit: y = {w_abs:.2f}x + {b_abs:.2f}")
for i, nu_fixed in enumerate(nu_values_fixed):
    w_t, b_t, sigma2_t = student_t_fits_fixed_nu[nu_fixed]
    plt.plot(x_line, w_t * x_line + b_t, linewidth=3, color=f"C{3+i}",
             label=fr"Student-t fit ($\nu={nu_fixed}$): y = {w_t:.2f}x + {b_t:.2f}")
plt.plot(x_line, w_ecme.item() * x_line + b_ecme.item(), linewidth=3, color="C6", linestyle="--",
         label=fr"Student-t fit (learned $\hat\nu={nu_ecme:.2f}$): y = {w_ecme.item():.2f}x + {b_ecme.item():.2f}")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Optimal line: SSE vs. Absolute loss vs. Student-t (fixed and learned $\nu$)")
plt.legend()
plt.grid(True)
plt.show()


#### Comparing convergence speed: gradient descent vs. EM

Let's compare the coordinate descent with the EM algorithm. First we only compare the case in which we fit the weights, by keeping the values of $\nu=1,\sigma^2=1$. We compare how many iterations each one needs to get close to the same cost value (we use as reference the loss that EM converges to).

In [ ]:
nu, sigma2 = 1, 1

# shared initialization: the OLS solution
w0, b0 = create_computation_graph_linear_from_ols(x_data_gd, t_data_gd)

n_epochs = 500

## gradient descent
w, b = w0.copy(), b0.copy()
lr = 0.001
gd_loss_history = []

for e in range(n_epochs):
    y_pred = computation_graph_linear(x_data_gd, w, b)
    gd_loss_history.append(np.sum(student_t_loss_function(t_data_gd, y_pred, nu, sigma2)))

    grad_w, grad_b, _, _ = grad_student_t_loss_wrt_linear_model(x_data_gd, t_data_gd, w, b, nu, sigma2)
    w = w - lr * grad_w
    b = b - lr * grad_b

## EM
w, b = w0.copy(), b0.copy()
em_loss_history = []

for e in range(n_epochs):
    y_pred = computation_graph_linear(x_data_gd, w, b)
    em_loss_history.append(np.sum(student_t_loss_function(t_data_gd, y_pred, nu, sigma2)))

    pi = student_t_e_step(x_data_gd, t_data_gd, w, b, nu, sigma2)
    w, b = student_t_m_step_w(x_data_gd, t_data_gd, pi)

plt.figure(figsize=(7, 5))
plt.plot(gd_loss_history, label="Gradient descent")
plt.plot(em_loss_history, label="EM")
plt.xlabel("Iteration")
plt.ylabel("Loss (Student-t)")
plt.title("Convergence speed: gradient descent vs. EM")
plt.legend()
plt.grid(True)
plt.show()

#### Comparing convergence speed: gradient descent vs. EM, optimizing $\wvec,\sigma^2,\nu$ together

Now let's compare both algorithms when all three parameters are learned jointly, instead of keeping $\nu,\sigma^2$ fixed as before.

In [ ]:
nu_init, sigma2_init = 1.0, 1.0
n_epochs = 500
lr_all = 0.001

# nu and sigma2 must stay strictly positive for the Student-t density to even be defined
# (nu is the degrees of freedom, sigma2 is a variance); unlike w,b, unconstrained gradient
# descent has no guarantee of respecting this, so after every step we clip them back into
# the valid region to avoid stepping into a domain where the loss is undefined (NaN/log of
# a non-positive number)
eps = 1e-3

## gradient descent, jointly on w,b,sigma2,nu
w, b = w0.copy(), b0.copy()
nu, sigma2 = nu_init, sigma2_init
gd_all_loss_history = []

for e in range(n_epochs):
    y_pred = computation_graph_linear(x_data_gd, w, b)
    gd_all_loss_history.append(np.sum(student_t_full_loss_function(t_data_gd, y_pred, nu, sigma2)))

    grad_w, grad_b, grad_sigma2, grad_nu = grad_student_t_loss_wrt_linear_model(x_data_gd, t_data_gd, w, b, nu, sigma2)
    w = w - lr_all * grad_w
    b = b - lr_all * grad_b
    sigma2 = sigma2 - lr_all * grad_sigma2
    nu = nu - lr_all * grad_nu

    # keep nu,sigma2 inside the valid domain of the Student-t density
    nu = max(nu, eps)
    sigma2 = max(sigma2, eps)

## EM, jointly on w,b,sigma2,nu (E-step once per iteration, reused by the two frozen-weight
## M-steps; nu's M-step is the only one maximizing the actual likelihood directly)
w, b = w0.copy(), b0.copy()
nu, sigma2 = nu_init, sigma2_init
em_all_loss_history = []

for e in range(n_epochs):
    y_pred = computation_graph_linear(x_data_gd, w, b)
    em_all_loss_history.append(np.sum(student_t_full_loss_function(t_data_gd, y_pred, nu, sigma2)))

    pi = student_t_e_step(x_data_gd, t_data_gd, w, b, nu, sigma2)
    w, b = student_t_m_step_w(x_data_gd, t_data_gd, pi)
    sigma2 = student_t_m_step_sigma2(x_data_gd, t_data_gd, w, b, pi)
    nu = student_t_m_step_nu(x_data_gd, t_data_gd, w, b, sigma2)

plt.figure(figsize=(7, 5))
plt.plot(gd_all_loss_history, label="Gradient descent")
plt.plot(em_all_loss_history, label="EM")
plt.xlabel("Iteration")
plt.ylabel("Loss (Student-t, full)")
plt.title(r"Convergence speed: gradient descent vs. EM (learning $w,\sigma^2,\nu$)")
plt.legend()
plt.grid(True)
plt.show()

### Final Remark

Importantly, we can also put linear models of basis functions on the parameters of the student-t or laplace distributions. One thing are the statistical assumptions of the distribution that has generate our data. The other, is how we paremterize its parameters. It can be with a linear model, with a linear basis function model, with a neural network and so on.

## Probabilistic Interpretation of Regularizers

We already saw in the MAP section that we have the option of placing a prior distribution over the parameters and maximizing the maximum posterior distribution. The question is, well, what if we place a prior distribution over the parameters of our models?.

In the same way we have seen that cost functions can be associated with performing maximum likelihood over some distribution (the squared loss with a Gaussian, the absolute loss with a Laplace, the Student-t loss with a Student-t, and so on), we can take that same likelihood $p(\mathcal{D}\mid\wvec)$, place a prior $p(\wvec)$ over its parameters, and define the resulting posterior via Bayes' rule:

$$
p(\wvec\mid\mathcal{D}) = \frac{p(\mathcal{D}\mid\wvec)\,p(\wvec)}{p(\mathcal{D})} \propto p(\mathcal{D}\mid\wvec)\,p(\wvec)
$$

where $p(\mathcal{D})=\int p(\mathcal{D}\mid\wvec)p(\wvec)\,d\wvec$ is a normalizing constant that does not depend on $\wvec$, so it can be dropped when maximizing over $\wvec$ (this is exactly the MAP estimate from before). If we maximize the log posterior distribution (again, log is monotonic so does not change the place where the optimum occurs), this would be equivalent to:

$$
\argmax_\wvec \log p(\wvec\mid\mathcal{D}) = \argmax_\wvec  \frac{p(\mathcal{D}\mid\wvec)\,p(\wvec)}{p(\mathcal{D})}  = \argmax_\wvec \log p(\mathcal{D}\mid\wvec)+\log p(\wvec)
$$

We have already derived many variants of the log-likelihood function $\log p(\mathcal{D}\mid\wvec)$, depending on the data we have. The next point is to analyze $\log p(\wvec)$. Remember that maximizing is equivalent to minimizing this loss multiplied by -1. In the same way we minimize $-\log p(\mathcal{D}\mid\wvec)$, which is equivalent to minimizing some of the studied loss functions, we are interested in minimizing $-\log p(\wvec)$.

### $L_2$ regularization: the Gaussian prior

Assume an isotropic Gaussian prior over $\wvec\in\mathbb{R}^D$, independent across coordinates and centered at zero, with the same variance $\tau^2$ for every coordinate:

$$
p(\wvec) = \mathcal{N}(\wvec\mid \mathbf{0},\tau^2\Imat) = \prod^D_{d=1}\frac{1}{\sqrt{2\pi\tau^2}}\exp\pare{-\frac{w_d^2}{2\tau^2}}
$$


Taking the negative log:

$$
\begin{split}
-\log p(\wvec) &= \sum^D_{d=1}\pare{\frac12\log(2\pi\tau^2) + \frac{w_d^2}{2\tau^2}}\\
&= \frac{D}{2}\log(2\pi\tau^2) + \frac{1}{2\tau^2}\sum^D_{d=1}w_d^2\\
&= \frac{1}{2\tau^2}||\wvec||_2^2 + \text{const}
\end{split}
$$

where "const" collects everything that does not depend on $\wvec$ and can be dropped when optimizing. Plugging this into the general result above, for any cost function $C(\wvec)$ coming from a log likelihood:

$$
\begin{split}
& \argmax_\wvec \log p(\wvec\mid\mathcal{D}) = \\
& \argmin_\wvec C(\wvec) + R(\wvec) = \\
& \argmin_\wvec C(\wvec) + \lambda||\wvec||_2^2, \qquad \lambda=\frac{1}{2\tau^2}
\end{split}
$$

which is exactly $L_2$ regularization: whether $C(\wvec)$ is the squared loss of a linear model, the cross-entropy of a logistic regression, or the loss of a deep network, adding a zero-mean Gaussian prior over the weights always contributes this same $\lambda||\wvec||_2^2$ term. A small prior variance $\tau^2$ (a confident, narrow prior around zero) gives a large $\lambda$, ie strong regularization; a large $\tau^2$ (a vague prior) gives a small $\lambda$; as $\tau^2\to\infty$ the prior becomes flat and $\lambda\to0$, recovering the plain, unregularized minimization of $C(\wvec)$.

This probabilistic interpretation is very clean. If we have a narrow Gaussian, concentrated at zero, this means that there is a lot of chance (from our perspective) that the parameter takes the value of 0. This is turned into a big $\lambda$, which implies that the optimization will easily drive the parameters towards zero, as we know.

### $L_1$ regularization: the Laplace prior

Assume instead an independent Laplace prior over each coordinate of $\wvec$, centered at zero with scale $b$:

$$
p(\wvec)=\prod^D_{d=1}\frac{1}{2b}\exp\pare{-\frac{|w_d|}{b}}
$$

Taking the negative log:

$$
\begin{split}
-\log p(\wvec) &= \sum^D_{d=1}\pare{\log(2b) + \frac{|w_d|}{b}}\\
&= D\log(2b) + \frac1b\sum^D_{d=1}|w_d|\\
&= \frac1b||\wvec||_1 + \text{const}
\end{split}
$$

where "const" collects everything that does not depend on $\wvec$ and can be dropped when optimizing. Plugging this into the general result above, for any cost function $C(\wvec)$ coming from a log likelihood:

$$
\begin{split}
& \argmax_\wvec \log p(\wvec\mid\mathcal{D}) = \\
& \argmin_\wvec C(\wvec) + R(\wvec) = \\
& \argmin_\wvec C(\wvec) + \lambda||\wvec||_1, \qquad \lambda=\frac1b
\end{split}
$$

which is exactly $L_1$ regularization: whether $C(\wvec)$ is the squared loss of a linear model, the cross-entropy of a logistic regression, or the loss of a deep network, adding a zero-mean Laplace prior over the weights always contributes this same $\lambda||\wvec||_1$ term. A small prior scale $b$ (a confident, narrow prior around zero) gives a large $\lambda$, ie strong regularization; a large $b$ (a vague prior) gives a small $\lambda$; as $b\to\infty$ the prior becomes flat and $\lambda\to0$, recovering the plain, unregularized minimization of $C(\wvec)$.

This probabilistic interpretation is very clean. If we have a narrow Laplace, concentrated at zero, this means that there is a lot of chance (from our perspective) that the parameter takes the value of 0. This is turned into a big $\lambda$, which implies that the optimization will easily drive the parameters towards zero, as we know.

### Visualizing prior densities

We now visualize the densities, observing how the mass placed by both priors corresponds exactly with the norm-constrained viewpoint of these regularizers


In [ ]:
%matplotlib inline
plt.close("all")

w_grid = np.linspace(-4, 4, 400)
w1, w2 = np.meshgrid(w_grid, w_grid)

tau2 = 1.0
gaussian_density = np.exp(-(w1**2 + w2**2) / (2 * tau2)) / (2 * np.pi * tau2)

b = 1.0 / np.sqrt(2.0)  # matches variance 1, same as tau^2=1 above, for a fair comparison
laplace_density = np.exp(-(np.abs(w1) + np.abs(w2)) / b) / (4 * b**2)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(w_grid, scipy_norm.pdf(w_grid, scale=np.sqrt(tau2)), color="C0")
axes[0, 0].set_xlabel(r"$w$")
axes[0, 0].set_ylabel(r"$p(w)$")
axes[0, 0].set_title(r"Gaussian prior, $\mathcal{N}(0,\tau^2)$")
axes[0, 0].grid(True)

axes[0, 1].contour(w1, w2, gaussian_density, levels=8, cmap="viridis")
axes[0, 1].set_xlabel(r"$w_1$")
axes[0, 1].set_ylabel(r"$w_2$")
axes[0, 1].set_title(r"Level sets of $p(w_1,w_2)$: circles, ie $L_2$ balls")
axes[0, 1].set_aspect("equal")
axes[0, 1].grid(True)

axes[1, 0].plot(w_grid, scipy_norm.pdf(w_grid, scale=np.sqrt(tau2)), color="C0", label="Gaussian")
axes[1, 0].plot(w_grid, scipy_laplace.pdf(w_grid, scale=b), color="C1", label="Laplace")
axes[1, 0].set_xlabel(r"$w$")
axes[1, 0].set_ylabel(r"$p(w)$")
axes[1, 0].set_title("Same variance, different shape")
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].contour(w1, w2, laplace_density, levels=8, cmap="viridis")
axes[1, 1].set_xlabel(r"$w_1$")
axes[1, 1].set_ylabel(r"$w_2$")
axes[1, 1].set_title(r"Level sets of $p(w_1,w_2)$: diamonds, ie $L_1$ balls")
axes[1, 1].set_aspect("equal")
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

### Why does $L_1$ give exact zeros?

The plot above already hints at the answer: the Laplace density has a sharp, non-differentiable peak exactly at $0$, unlike the smooth Gaussian, so it places comparatively more prior mass exactly at $w=0$.

We already saw the mechanism behind this in the [$L_1$ regularization section](2_Linear_basis_function_models_ovefit_reg.ipynb) of the regression chapter, via the gradient of the regularized loss: the $L_2$ penalty contributes $2\lambda w$, a force that vanishes as $w\to0$, so the data term always wins by a little and $w_\text{opt}$ only reaches exactly $0$ in the limit $\lambda\to\infty$. The $L_1$ penalty instead contributes $\lambda\,\text{sgn}(w)$, a force of constant magnitude $\lambda$ regardless of how close $w$ already is to $0$; when the pull from the data is weaker than this constant force, $w_\text{opt}$ gets stuck exactly at $0$, for an entire range of the data, not just in some limit. Making precise exactly when that happens requires the notion of a subgradient, which we have not introduced in this book.

### New prior densities

In the same way we have designed new loss functions coming from likelihoods that had some nice properties for our interests, we can do the same with regularizers. If, for whatever reason, we want our prior distribution over the parameters to be centered somewhere different from 0, we can just plug in the desired target mean into the priors and derive the associated regularizer. Same if we do not want the prior to factorize and establish dependencies across points.

Also, we have only shown the connection between regularizers and the prior over $\wvec$. However, since we now know that loss functions might come from probability distributions, we might be interested in placing priors over other parameters of the likelihood such as the noise $\sigma^2$ or the degrees of freedom. We just need to write down the joint prior $p(\wvec,\sigma^2)$ and work out the corresponding loss function. These joint priors might or might not factorize. This is up to you and your interests.

## TODO

* When possible, extend derivations through matrix normal distributions
* Heteroscedastic multioutput Gaussian model through matrix normal distribution.
* Heteroscedastic Gaussian model with shared weights for variance and mean models.
* Heteroscedastic and multioutput models for Student-t and Laplace.
* Examples of linear basis function models applied to Student-t.
* Talk about the distribution of residuals of each model.
* Read some of the assumptions from the econometrics viewpoint of this part and establish connections. Things such as the expected value of the noise given X being 0 and so on.